<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/triton_Voxtral_topo_cbp_gguf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## setup

In [ ]:
!pip install -U bitsandbytes>=0.46.1 -q

In [ ]:
!nvidia-smi

Tue Nov 24 01:33:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   72C    P8             20W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!ls -ltha /content/drive/MyDrive/GGUF_Models/

total 452G
drwx------ 4 root root 4.0K Nov 24 00:10 ..
-rw------- 1 root root  180 Nov 23 15:09 granite_4_0_h_small_q8_0_audit.json
-rw------- 1 root root  32G Nov 23 14:42 granite-4.0-h-small-topo-triton-Q8_0-patched.gguf
-rw------- 1 root root  32G Nov 23 11:42 granite-4.0-h-small-topo-triton-Q8_0.gguf
-rw------- 1 root root  16G Nov 23 05:21 kimi-vl-ft-Q8_0.gguf
-rw------- 1 root root 3.7K Nov 22 15:34 GLM-4-32B-0414-topo-triton-Q8_0_audit.json
-rw------- 1 root root  33G Nov 22 14:19 GLM-4-32B-0414-topo-triton-Q8_0.gguf
-rw------- 1 root root 8.9K Nov 21 14:30 somala_drive_gguf_audit.json
-rw------- 1 root root  26G Nov 21 05:08 Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0.gguf
-rw------- 1 root root 7.2G Nov 19 03:18 mistral-7b-text-to-sql-topo-Q8_0.gguf
-rw------- 1 root root  27G Nov 14 20:14 qwen3.8-27b-topocbp-Q8_0.gguf
-rw------- 1 root root  11G Nov 12 06:55 deepseek_moe_16b_topocbp_q4_k_m.gguf
-rw------- 1 root root  70G Nov  9 06:01 deepseek-r1-distill-llama-70b-topo-governed-

In [ ]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu130 \
  --no-cache-dir

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu130


In [ ]:
!pip install mistral-common

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 32.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 70.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [mistral-common]


In [ ]:
!pip show transformers llama-cpp-python bitsandbytes mistral-common

Name: transformers
Version: 5.22.1
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.13/dist-packages
Requires: huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer
Required-by: peft, sentence-transformers
---
Name: llama_cpp_python
Version: 0.3.38
Summary: Python bindings for the llama.cpp library
Home-page: https://github.com/abetlen/llama-cpp-python
Author: 
Author-email: Andrei Betlen <abetlen@gmail.com>
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: diskcache, jinja2, numpy, typing-extension

## train

In [ ]:
import os
import logging
import warnings

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("HF_HUB_CACHE", "/content/hf_cache")
warnings.filterwarnings("ignore", category=FutureWarning, message=r".*rope_scaling.*")
warnings.filterwarnings("ignore", message=r".*large amount of egress.*")
warnings.filterwarnings("ignore", message=r".*inner dimension.*not aligned.*")

import gc
import math
import hashlib
import random
import torch
import torch.nn.functional as F
from torch.optim import Optimizer
from torch.utils.data import DataLoader
import transformers
from packaging.version import Version
from transformers import (
    AutoConfig,
    AutoProcessor,
    BitsAndBytesConfig,
    VoxtralRealtimeForConditionalGeneration,
    get_cosine_schedule_with_warmup,
)
from transformers.utils import logging as hf_logging
from datasets import Audio, load_dataset

hf_logging.set_verbosity_error()
logging.getLogger("huggingface_hub.utils._http").setLevel(logging.ERROR)

try:
    import triton
    import triton.language as tl

    HAS_TRITON = True
except ImportError:
    triton = None
    tl = None
    HAS_TRITON = False

# =====================================================================
# 0. SEED & DEVICE
# =====================================================================
SEED = 123
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
use_cuda = num_gpus > 0
if not use_cuda:
    raise RuntimeError("4-bit bitsandbytes loading requires a CUDA GPU.")
primary_device = torch.device("cuda:0")

# =====================================================================
# 1. OPTIMIZER: TopoAdamW with bf16 states, stochastic rounding and
#    error-compensated bf16 parameter updates (fused Triton kernel +
#    PyTorch fallback). Optional frozen "anchor" rows in the embedding.
#    NOTE: the anchor audit compares weights against a clone taken from
#    the same weights, so it is a freeze check, not an external certificate.
# =====================================================================
PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]


def _stochastic_round_(src_fp32: torch.Tensor, dst_bf16: torch.Tensor) -> None:
    """Round fp32 -> bf16 stochastically. NOTE: src_fp32 is modified in place."""
    bits = src_fp32.view(torch.int32)
    bits.add_(torch.randint_like(bits, 0, 1 << 16))
    bits.bitwise_and_(-65536)
    dst_bf16.copy_(src_fp32)


if HAS_TRITON:

    @triton.jit
    def _topo_adamw_fused_step_kernel(
        p_ptr,
        grad_ptr,
        m_buf_ptr,
        v_buf_ptr,
        comp_ptr,
        n_elements,
        lr,
        beta1,
        beta2,
        eps,
        wd,
        bias_corr1,
        bias_corr2,
        seed_m,
        seed_v,
        P_IS_BF16: tl.constexpr,
        BLOCK_SIZE: tl.constexpr,
    ):
        pid = tl.program_id(axis=0)
        offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
        mask = offsets < n_elements

        p_raw = tl.load(p_ptr + offsets, mask=mask, other=0.0)
        g = tl.load(grad_ptr + offsets, mask=mask, other=0.0).to(tl.float32)
        m_prev = tl.load(m_buf_ptr + offsets, mask=mask, other=0.0).to(tl.float32)
        v_prev = tl.load(v_buf_ptr + offsets, mask=mask, other=0.0).to(tl.float32)
        comp_prev = tl.load(comp_ptr + offsets, mask=mask, other=0.0).to(tl.float32)

        w = p_raw.to(tl.float32)

        m = beta1 * m_prev + (1.0 - beta1) * g
        v = beta2 * v_prev + (1.0 - beta2) * g * g

        m_hat = m / bias_corr1
        v_hat = v / bias_corr2
        denom = tl.sqrt(v_hat) + eps
        step_update = -lr * (m_hat / denom + wd * w) + comp_prev

        total = w + step_update
        if P_IS_BF16:
            new_p = total.to(tl.bfloat16)
            new_comp = total - new_p.to(tl.float32)
            tl.store(p_ptr + offsets, new_p, mask=mask)
            tl.store(comp_ptr + offsets, new_comp.to(tl.bfloat16), mask=mask)
        else:
            tl.store(p_ptr + offsets, total, mask=mask)
            tl.store(comp_ptr + offsets, tl.zeros_like(total).to(tl.bfloat16), mask=mask)

        m_bits = m.to(tl.int32, bitcast=True)
        m_jitter = (tl.randint(seed_m, offsets) & 0xFFFF).to(tl.int32)
        m_rounded_bits = (m_bits + m_jitter) & -65536
        m_rounded = m_rounded_bits.to(tl.float32, bitcast=True).to(tl.bfloat16)

        v_bits = v.to(tl.int32, bitcast=True)
        v_jitter = (tl.randint(seed_v, offsets) & 0xFFFF).to(tl.int32)
        v_rounded_bits = (v_bits + v_jitter) & -65536
        v_rounded = v_rounded_bits.to(tl.float32, bitcast=True).to(tl.bfloat16)

        tl.store(m_buf_ptr + offsets, m_rounded, mask=mask)
        tl.store(v_buf_ptr + offsets, v_rounded, mask=mask)


def _triton_param_step(
    p,
    grad,
    m_buf,
    v_buf,
    comp,
    lr,
    beta1,
    beta2,
    eps,
    wd,
    step_t,
    base_seed=SEED,
    param_id=0,
    block_size=1024,
) -> None:
    n_elements = p.numel()
    if n_elements == 0:
        return

    p_c = p.contiguous().view(-1)
    g_c = grad.contiguous().view(-1)
    m_c = m_buf.contiguous().view(-1)
    v_c = v_buf.contiguous().view(-1)
    c_c = comp.contiguous().view(-1)

    bias_corr1 = 1.0 - (beta1**step_t)
    bias_corr2 = 1.0 - (beta2**step_t)
    seed_m = int((base_seed * 1_000_003 + step_t * 97 + param_id * 7919) & 0x7FFFFFFF)
    seed_v = int((base_seed * 1_000_033 + step_t * 193 + param_id * 104729) & 0x7FFFFFFF)

    grid = (triton.cdiv(n_elements, block_size),)
    _topo_adamw_fused_step_kernel[grid](
        p_c,
        g_c,
        m_c,
        v_c,
        c_c,
        n_elements,
        float(lr),
        float(beta1),
        float(beta2),
        float(eps),
        float(wd),
        float(bias_corr1),
        float(bias_corr2),
        seed_m,
        seed_v,
        P_IS_BF16=(p_c.dtype == torch.bfloat16),
        BLOCK_SIZE=block_size,
    )

    if not p.is_contiguous():
        p.copy_(p_c.view_as(p))
    if not m_buf.is_contiguous():
        m_buf.copy_(m_c.view_as(m_buf))
    if not v_buf.is_contiguous():
        v_buf.copy_(v_c.view_as(v_buf))
    if not comp.is_contiguous():
        comp.copy_(c_c.view_as(comp))


def _pytorch_param_step(p, m_buf, v_buf, comp, lr, beta1, beta2, eps, wd, step_t) -> None:
    g = p.grad.float()
    m = m_buf.float().mul_(beta1).add_(g, alpha=1.0 - beta1)
    v = v_buf.float().mul_(beta2).addcmul_(g, g, value=1.0 - beta2)
    del g

    w = p.detach().float().clone()
    denom = (v.sqrt() / math.sqrt(1.0 - beta2**step_t)).add_(eps)
    y = m.div(denom).mul_(-lr / (1.0 - beta1**step_t))
    del denom

    y.add_(w, alpha=-lr * wd)
    if p.dtype == torch.bfloat16:
        y.add_(comp.float())
    _stochastic_round_(m, m_buf)
    _stochastic_round_(v, v_buf)
    del m, v

    total = w.add_(y)
    if p.dtype == torch.bfloat16:
        new = total.to(torch.bfloat16)
        comp.copy_(total.sub_(new.float()))
        p.copy_(new)
    else:
        p.copy_(total.to(p.dtype))
        comp.zero_()
    del y, w, total


class TopoAdamW(Optimizer):
    """AdamW with bf16 moments (stochastic rounding) and compensated bf16 params."""

    def __init__(
        self,
        params,
        lr=1e-4,
        betas=(0.9, 0.999),
        eps=1e-8,
        weight_decay=1e-2,
        embedding_layer=None,
        prime_anchors=None,
        base_seed=SEED,
        use_triton=True,
    ):
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay))
        self.base_seed = base_seed
        self.use_triton = bool(use_triton and HAS_TRITON)
        self.embedding_layer = embedding_layer

        self._param_ids = {}
        for g in self.param_groups:
            for p in g["params"]:
                self._param_ids[id(p)] = len(self._param_ids)

        self.anchors_active = False
        self.valid_anchors = []
        self.baseline = {}
        self.certified_sha256_digest = None

        if embedding_layer is not None and hasattr(embedding_layer, "weight"):
            emb_w = embedding_layer.weight
            optimized = any(p is emb_w for g in self.param_groups for p in g["params"])
            if optimized:
                anchors = list(prime_anchors) if prime_anchors is not None else list(PRIME_ANCHORS)
                self.valid_anchors = [i for i in anchors if i < emb_w.shape[0]]
                self.baseline = {i: emb_w.data[i].detach().clone() for i in self.valid_anchors}
                self.certified_sha256_digest = self._digest()
                self.anchors_active = bool(self.valid_anchors)

        print(
            f"[TopoAdamW] Triton={self.use_triton} | anchors_active={self.anchors_active}"
            + (f" | digest {self.certified_sha256_digest[:16]}" if self.anchors_active else "")
        )

    def _digest(self) -> str:
        w = self.embedding_layer.weight.data
        buf = torch.cat([w[i].flatten().float() for i in self.valid_anchors]).cpu().numpy().tobytes()
        return hashlib.sha256(buf).hexdigest()

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        emb_w = self.embedding_layer.weight if self.anchors_active else None
        if emb_w is not None and emb_w.grad is not None:
            for i in self.valid_anchors:
                emb_w.grad[i].zero_()

        for group in self.param_groups:
            lr, wd, eps = group["lr"], group["weight_decay"], group["eps"]
            beta1, beta2 = group["betas"]
            for p in group["params"]:
                if not p.requires_grad or p.grad is None:
                    continue
                state = self.state[p]
                if not state:
                    state["step"] = 0
                    state["exp_avg"] = torch.zeros_like(p, dtype=torch.bfloat16)
                    state["exp_avg_sq"] = torch.zeros_like(p, dtype=torch.bfloat16)
                    state["comp"] = torch.zeros_like(p, dtype=torch.bfloat16)

                state["step"] += 1
                t = state["step"]
                m_buf, v_buf, comp = state["exp_avg"], state["exp_avg_sq"], state["comp"]
                grad_on_dev = p.grad.to(p.device) if p.grad.device != p.device else p.grad

                if self.use_triton and p.is_cuda and grad_on_dev.is_cuda:
                    _triton_param_step(
                        p,
                        grad_on_dev,
                        m_buf,
                        v_buf,
                        comp,
                        lr,
                        beta1,
                        beta2,
                        eps,
                        wd,
                        t,
                        base_seed=self.base_seed,
                        param_id=self._param_ids[id(p)],
                    )
                else:
                    orig_grad = p.grad
                    p.grad = grad_on_dev
                    try:
                        _pytorch_param_step(p, m_buf, v_buf, comp, lr, beta1, beta2, eps, wd, t)
                    finally:
                        p.grad = orig_grad

                if emb_w is not None and p is emb_w:
                    for i in self.valid_anchors:
                        p[i].copy_(self.baseline[i].to(device=p.device, dtype=p.dtype))
                        comp[i].zero_()

        return loss

    def audit_telemetry(self):
        """Returns (max_drift, digest, matched), or (None, None, None) if anchoring is inactive."""
        if not self.anchors_active:
            return None, None, None
        w = self.embedding_layer.weight.data
        drift = max(
            torch.max(torch.abs(w[i].float() - self.baseline[i].to(w.device).float())).item()
            for i in self.valid_anchors
        )
        h = self._digest()
        return drift, h, h == self.certified_sha256_digest


def selftest_optimizer():
    if not (HAS_TRITON and torch.cuda.is_available()):
        print("[selftest] skipped (needs Triton + CUDA)")
        return
    for dtype in (torch.bfloat16, torch.float32):
        torch.manual_seed(0)
        shape = (257, 129)
        p0 = torch.randn(shape, device="cuda").to(dtype)
        g = torch.randn(shape, device="cuda").to(torch.bfloat16)
        outs = []
        for use_t in (True, False):
            p = p0.clone()
            p.grad = g.clone()
            m = torch.zeros(shape, device="cuda", dtype=torch.bfloat16)
            v = torch.zeros_like(m)
            c = torch.zeros_like(m)
            if use_t:
                _triton_param_step(p, g, m, v, c, 1e-3, 0.9, 0.999, 1e-8, 1e-2, 1)
            else:
                _pytorch_param_step(p, m, v, c, 1e-3, 0.9, 0.999, 1e-8, 1e-2, 1)
            outs.append(p.float())
        diff = (outs[0] - outs[1]).abs().max().item()
        print(f"[selftest] {dtype}: max |triton - pytorch| = {diff:.3e}")


if os.environ.get("RUN_OPT_SELFTEST") == "1":
    selftest_optimizer()

# =====================================================================
# STREAMING-LAYOUT LOGIC (pure python / numpy: no torch, unit-testable)
#
# Voxtral Realtime decodes ONE token per 80 ms audio step. At step t the decoder
# input is  embed(previous emitted token) + audio_embedding[t]  (see the
# `inputs_embeds += audio_embeds` line in the model), so input_ids must have exactly
# as many positions as there are audio steps.
#
# Target stream y[j] (what the model emits at step j):
#   * [STREAMING_PAD] while no word is complete (or a word is still being spoken)
#   * once a word has ended AND `delay` steps have passed: [STREAMING_WORD] followed by
#     the word's subword tokens, one per step
#   * words that collide with tokens still being emitted are appended directly
#     (no extra [STREAMING_WORD])
# input_ids = [BOS] + y[:-1]   (the prompt BOS + pads is exactly the first P positions)
# labels[j] = y[j]  (already aligned to the output position; NO extra shift needed)
# =====================================================================
import re

import numpy as np

CTC_FRAME_SEC = 0.02  # wav2vec2 emits one frame per 20 ms
_ALIGN_STRIP = re.compile(r"[^A-Z']")


def split_transcript_words(text):
    """Split into display words and 'clean' words used for CTC alignment.

    clean word = uppercase letters/apostrophes only. Pure punctuation tokens are glued to the
    previous word. Tokens with no letters but with digits are kept as their own word with an
    empty clean form (they are not alignable, so they inherit the previous word's end time).
    """
    display, clean = [], []
    for raw_w in text.split():
        c = _ALIGN_STRIP.sub("", raw_w.upper())
        if c:
            display.append(raw_w)
            clean.append(c)
        elif any(ch.isalnum() for ch in raw_w):
            display.append(raw_w)
            clean.append("")
        elif display:
            display[-1] = display[-1] + raw_w
        # else: leading pure punctuation -> dropped
    return display, clean


def ctc_word_end_frames(emission, clean_words, vocab, blank):
    """Forced-align `clean_words` to CTC log-probs `emission` (T, V).

    Simplified CTC trellis (as in the torchaudio forced-alignment tutorial): a token is
    emitted on exactly one frame, every other frame emits blank. Returns, per word, the frame
    index at which its LAST character was emitted (None for words with no alignable chars),
    or None if alignment is impossible.
    """
    emission = np.asarray(emission, dtype=np.float64)
    tokens, owner = [], []
    aligned_idx = [i for i, w in enumerate(clean_words) if w]
    if not aligned_idx:
        return None
    for n, i in enumerate(aligned_idx):
        if n > 0:
            tokens.append(vocab["|"])
            owner.append(-1)
        for ch in clean_words[i]:
            tokens.append(vocab[ch])
            owner.append(i)

    T, J = emission.shape[0], len(tokens)
    if J == 0 or T < J:
        return None
    stay = emission[:, blank]
    tok = emission[:, tokens]  # (T, J)

    trellis = np.full((T + 1, J + 1), -np.inf, dtype=np.float64)
    trellis[0, 0] = 0.0
    for t in range(T):
        trellis[t + 1, 0] = trellis[t, 0] + stay[t]
        trellis[t + 1, 1:] = np.maximum(trellis[t, 1:] + stay[t], trellis[t, :-1] + tok[t])
    if not np.isfinite(trellis[T, J]):
        return None

    emit = [-1] * J
    t, j = T, J
    while j > 0 and t > 0:
        stayed = trellis[t - 1, j] + stay[t - 1]
        changed = trellis[t - 1, j - 1] + tok[t - 1, j - 1]
        if changed > stayed:
            emit[j - 1] = t - 1
            j -= 1
        t -= 1
    if j > 0:
        return None

    ends = [None] * len(clean_words)
    for k, o in enumerate(owner):
        if o >= 0:
            ends[o] = emit[k] if ends[o] is None else max(ends[o], emit[k])
    return ends


def frames_to_end_secs(end_frames):
    """CTC frame indices -> seconds, forward-filling words that could not be aligned."""
    out, last = [], 0.0
    for f in end_frames:
        sec = last if f is None else (f + 1) * CTC_FRAME_SEC
        out.append(sec)
        last = sec
    return out


def uniform_word_ends(words, duration_sec):
    """Fallback when alignment fails: spread word end times proportionally to word length."""
    weights = np.array([max(1, len(w)) for w in words], dtype=np.float64)
    return (duration_sec * np.cumsum(weights) / weights.sum()).tolist()


def build_stream(word_ids, end_steps, n_steps, prompt_ids, pad_id, word_id, delay):
    """Build (input_ids, labels, n_truncated) for one utterance.

    word_ids  : list[list[int]]  subword ids per word
    end_steps : list[int]        decoder step (padded timeline) at which each word ends
    n_steps   : int              number of audio steps (= decoder positions)
    prompt_ids: [BOS] + [PAD]*(n_left + n_delay)
    """
    P = len(prompt_ids)
    y = [pad_id] * n_steps
    cursor = 0  # next free output position
    truncated = 0
    for ids, f_end in zip(word_ids, end_steps):
        start = f_end + delay
        if start > cursor:  # gap since the previous token -> new group starts with [W]
            if start >= n_steps:
                truncated += 1
                continue
            y[start] = word_id
            pos = start + 1
        else:  # collides with / directly follows the previous word -> no extra [W]
            pos = cursor
        for tok in ids:
            if pos >= n_steps:
                truncated += 1
                break
            y[pos] = tok
            pos += 1
        cursor = max(cursor, pos)

    input_ids = ([prompt_ids[0]] + y[: n_steps - 1])[:n_steps]
    labels = list(y)
    # The prompt already supplies the first P-1 emitted tokens (all pads): nothing to predict there.
    for j in range(min(P - 1, n_steps)):
        labels[j] = -100
    return input_ids, labels, truncated

# =====================================================================
# 2. MODEL / PROCESSOR (mistralai/Voxtral-Mini-4B-Realtime-2602)
# =====================================================================
import uuid
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

model_id = "mistralai/Voxtral-Mini-4B-Realtime-2602"

# trust_remote_code is not needed: the model class is native to transformers.
config = AutoConfig.from_pretrained(model_id)
config.use_cache = False

processor = AutoProcessor.from_pretrained(model_id)
tokenizer = processor.tokenizer  # MistralCommonBackend (the processor requires it)

has_prequant = getattr(config, "quantization_config", None) is not None
model_kwargs = dict(
    config=config,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)
if not has_prequant:
    # NOTE: skip patterns are matched against the FULL module path from the start (or as a
    # suffix), and the real paths are "model.audio_tower...", "model.multi_modal_projector...",
    # "model.time_embedding...", "lm_head". A bare "audio_tower" never matched, which is why
    # the audio tower used to end up 4-bit quantized.
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        llm_int8_skip_modules=[
            "lm_head",
            "model.audio_tower",
            "model.multi_modal_projector",
            "model.time_embedding",
        ],
    )

print(
    f"[*] transformers {transformers.__version__} | "
    f"model_type={getattr(config, 'model_type', 'unknown')}"
)
_dtype_kw = "dtype" if Version(transformers.__version__) >= Version("4.56.0") else "torch_dtype"
model = VoxtralRealtimeForConditionalGeneration.from_pretrained(
    model_id, **{_dtype_kw: torch.bfloat16}, **model_kwargs
)

model.config.use_cache = False
warnings.filterwarnings("ignore", message=r".*inner dimension.*not aligned.*")
gc.collect()
torch.cuda.empty_cache()

# Sanity check: confirm the skip-module names actually matched (audio tower should NOT be 4-bit).
try:
    import bitsandbytes as bnb

    n4_audio = sum(
        isinstance(m, bnb.nn.Linear4bit) for n, m in model.named_modules() if "audio_tower" in n
    )
    n4_total = sum(isinstance(m, bnb.nn.Linear4bit) for m in model.modules())
    print(f"[check] Linear4bit modules: total={n4_total}, inside audio_tower={n4_audio} (expect 0)")
    if n4_audio:
        print("[warn] audio_tower contains 4-bit layers; check llm_int8_skip_modules names.")
except Exception as e:
    print(f"[warn] 4-bit module check failed: {e!r}")

embedding_layer = (
    model.get_input_embeddings() if hasattr(model, "get_input_embeddings") else None
)

# ---- Choose trainable params for Voxtral Realtime ----
# Default: lm_head (tied with the token embedding) + norm weights.
# Set TRAIN_AUDIO_TOWER_AND_ADAPTER=1 to also train the (bf16) audio tower / projector
# linear layers (excluding conv frontends). Quantized (non-floating-point) weights are skipped.
train_audio_tower_and_adapter = os.environ.get("TRAIN_AUDIO_TOWER_AND_ADAPTER", "0") == "1"

for p in model.parameters():
    p.requires_grad = False

decay_params, no_decay_params, seen = [], [], set()
for name, mod in model.named_modules():
    is_conv = isinstance(mod, (torch.nn.Conv1d, torch.nn.Conv2d))
    if is_conv:
        continue
    is_head = name.endswith("lm_head")
    is_norm = "norm" in name.lower()
    is_audio_or_proj = train_audio_tower_and_adapter and any(
        tok in name.lower()
        for tok in ("audio", "projector", "adapter", "multi_modal", "cross_attn", "q_proj", "k_proj", "v_proj", "o_proj")
    )
    if not (is_head or is_norm or is_audio_or_proj):
        continue
    for p in mod.parameters(recurse=False):
        if id(p) in seen or not p.is_floating_point():
            continue
        seen.add(id(p))
        p.requires_grad = True
        use_decay = (is_head or is_audio_or_proj) and p.ndim >= 2
        (decay_params if use_decay else no_decay_params).append(p)

trainable_params = decay_params + no_decay_params
if not trainable_params:
    raise RuntimeError("No trainable parameters found; check Voxtral module names.")

if hasattr(model, "gradient_checkpointing_enable"):
    try:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    except Exception:
        model.gradient_checkpointing_enable()

try:
    model.enable_input_require_grads()
except Exception as e:
    print(f"[warn] enable_input_require_grads failed: {e!r}")

print(
    f"[*] Trainable parameters: {sum(p.numel() for p in trainable_params):,} "
    f"(train_audio_tower_and_adapter={train_audio_tower_and_adapter})"
)

WEIGHT_DECAY = 1e-2
topo_optimizer = TopoAdamW(
    [
        {"params": decay_params, "weight_decay": WEIGHT_DECAY},
        {"params": no_decay_params, "weight_decay": 0.0},
    ],
    lr=1e-4,
    embedding_layer=embedding_layer,
    prime_anchors=PRIME_ANCHORS,
    base_seed=SEED,
    use_triton=True,
)


def _get_input_device() -> torch.device:
    emb = embedding_layer
    if emb is not None and hasattr(emb, "weight") and emb.weight.device.type != "meta":
        return emb.weight.device
    return primary_device


input_device = _get_input_device()
print(f"[*] Input target device: {input_device}")

# =====================================================================
# 3. STREAMING LAYOUT CONSTANTS + SPEECH DATA & COLLATOR
# =====================================================================
# Everything below is derived from the processor/model so it follows the checkpoint.
_audio_cfg = processor.mistral_common_audio_config
n_left = int(_audio_cfg.n_left_pad_tokens)  # silence tokens the processor prepends to the audio
n_delay = int(processor.num_delay_tokens)  # words are emitted this many steps after they end
alen = int(getattr(model.config, "audio_length_per_tok", 8))  # mel frames per decoder step
hop = int(processor.feature_extractor.hop_length)
frame_sec = hop * alen / 16000.0  # 80 ms per decoder step

_probe_enc = processor(audio=np.zeros(16000, dtype=np.float32), sampling_rate=16000, return_tensors="pt")
prompt_ids = _probe_enc["input_ids"][0].tolist()  # [BOS] + [STREAMING_PAD] * (n_left + n_delay)
P = len(prompt_ids)
PAD_ID = prompt_ids[1]
if not (all(t == PAD_ID for t in prompt_ids[1:]) and P == 1 + n_left + n_delay):
    raise RuntimeError(f"Unexpected processor prompt layout: len={P}, n_left={n_left}, n_delay={n_delay}")

_tek = processor.tokenizer.tokenizer.instruct_tokenizer.tokenizer  # raw Tekkenizer
if not _tek.is_special("[STREAMING_WORD]"):
    raise RuntimeError("Tokenizer has no [STREAMING_WORD] token; cannot build streaming targets.")
W_ID = int(_tek.get_special_token("[STREAMING_WORD]"))
print(
    f"[*] stream layout: prompt_len={P} (n_left={n_left}, n_delay={n_delay}) | step={frame_sec * 1000:.0f} ms | "
    f"PAD_ID={PAD_ID} | WORD_ID={W_ID}"
)

audio_dataset_id = os.environ.get("VOXTRAL_AUDIO_DATASET", "hf-internal-testing/librispeech_asr_dummy")
audio_config_name = os.environ.get("VOXTRAL_AUDIO_CONFIG", None)
audio_split = os.environ.get("VOXTRAL_AUDIO_SPLIT", "validation")
ALIGNER_ID = os.environ.get("VOXTRAL_ALIGNER", "facebook/wav2vec2-base-960h")  # English CTC aligner
MAX_AUDIO_SEC = float(os.environ.get("VOXTRAL_MAX_AUDIO_SEC", "30"))
# LibriSpeech-style transcripts are ALL CAPS with no punctuation. By default we convert
# ALL-CAPS transcripts to sentence case so the model's output style is not pulled toward
# uppercase. Set VOXTRAL_NORMALIZE_TEXT=0 to disable.
normalize_text = os.environ.get("VOXTRAL_NORMALIZE_TEXT", "1") == "1"

_raw_cache_dir = os.path.join("/content/hf_cache", audio_dataset_id.replace("/", "__"))
os.makedirs(_raw_cache_dir, exist_ok=True)
raw = (
    load_dataset(audio_dataset_id, audio_config_name, split=audio_split, cache_dir=_raw_cache_dir)
    if audio_config_name
    else load_dataset(audio_dataset_id, split=audio_split, cache_dir=_raw_cache_dir)
)

audio_col = None
for cand in ("audio", "speech"):
    if cand in raw.column_names:
        audio_col = cand
        break
if audio_col is None:
    for col, feat in raw.features.items():
        if isinstance(feat, Audio) or "audio" in col.lower() or "speech" in col.lower():
            audio_col = col
            break
if audio_col is None:
    raise KeyError(f"Could not find an audio column in columns={raw.column_names}")

raw = raw.cast_column(audio_col, Audio(sampling_rate=16000))


def _extract_audio_array(sample):
    audio = sample.get(audio_col)
    if isinstance(audio, dict) and "array" in audio:
        return audio["array"], int(audio.get("sampling_rate") or 16000)
    if hasattr(audio, "get_all_samples"):
        decoded = audio.get_all_samples()
        arr = decoded.data.squeeze(0).detach().cpu().numpy()
        sr = int(getattr(decoded, "sample_rate", 16000))
        return arr, sr
    for arr_attr in ("array", "waveform"):
        if hasattr(audio, arr_attr):
            arr = getattr(audio, arr_attr)
            if hasattr(arr, "detach"):
                arr = arr.detach().cpu().numpy()
            sr = int(getattr(audio, "sampling_rate", getattr(audio, "sample_rate", 16000)))
            return arr, sr
    raise KeyError(
        f"Unsupported audio payload type={type(audio).__name__} in column={audio_col!r}; "
        f"sample keys={list(sample.keys())}"
    )


def _extract_transcript(sample) -> str:
    for col in ("text", "transcription", "sentence", "transcript"):
        val = sample.get(col)
        if isinstance(val, str) and val.strip():
            text = val.strip()
            if normalize_text and text.isupper():
                text = text.lower()
                text = text[:1].upper() + text[1:]
            return text
    return ""


class CTCWordAligner:
    """Word end times via CTC forced alignment (English, letters only).

    Voxtral Realtime targets are frame-synchronous, so every word needs a timestamp. If your
    dataset already has word timestamps, skip this and fill `word_end_sec` from your data.
    """

    def __init__(self, aligner_id, device):
        self.device = device
        self.proc = Wav2Vec2Processor.from_pretrained(aligner_id)
        self.model = Wav2Vec2ForCTC.from_pretrained(aligner_id).to(device).eval()
        self.vocab = {k.upper(): v for k, v in self.proc.tokenizer.get_vocab().items()}
        self.blank = self.proc.tokenizer.pad_token_id

    @torch.no_grad()
    def word_end_secs(self, wav, sr, clean_words):
        x = self.proc(np.asarray(wav, dtype=np.float32), sampling_rate=sr, return_tensors="pt").input_values
        logits = self.model(x.to(self.device)).logits[0].float()
        emission = torch.log_softmax(logits, dim=-1).cpu().numpy()
        frames = ctc_word_end_frames(emission, clean_words, self.vocab, self.blank)
        return None if frames is None else frames_to_end_secs(frames)


print(f"[*] loading CTC aligner {ALIGNER_ID} ...")
aligner = CTCWordAligner(ALIGNER_ID, input_device)


def add_alignment(sample):
    wav, sr = _extract_audio_array(sample)
    dur = len(wav) / float(sr)
    display, clean = split_transcript_words(_extract_transcript(sample))
    if not display:
        return {"words": [], "word_end_sec": [], "align_ok": False, "dur_sec": dur}
    ends = None
    try:
        ends = aligner.word_end_secs(wav, sr, clean)
    except Exception as e:  # e.g. a character missing from the aligner vocab
        print(f"[warn] alignment failed ({e!r}); using uniform timing for this sample")
    ok = ends is not None
    if not ok:
        ends = uniform_word_ends(display, dur)
    return {"words": display, "word_end_sec": [float(x) for x in ends], "align_ok": ok, "dur_sec": dur}


shuffled = raw.shuffle(seed=SEED)
n_total = len(shuffled)
n_train = max(1, min(12500, int(n_total * 0.9)) if n_total > 10 else max(1, n_total - 1))
n_eval = max(1, min(500, n_total - n_train)) if n_total > n_train else 1


def _prep(split):
    # explicit fingerprint: otherwise `datasets` tries to hash the whole aligner model
    ds = split.map(add_alignment, load_from_cache_file=False, new_fingerprint=uuid.uuid4().hex)
    return ds.filter(
        lambda words, dur: len(words) > 0 and dur <= MAX_AUDIO_SEC,
        input_columns=["words", "dur_sec"],
    )


train_ds = _prep(shuffled.select(range(n_train)))
eval_ds = _prep(shuffled.select(range(n_train, min(n_total, n_train + n_eval))))
print(f"[*] train samples: {len(train_ds)} | eval samples: {len(eval_ds)}")
_ok = sum(bool(x) for x in train_ds["align_ok"]) + sum(bool(x) for x in eval_ds["align_ok"])
print(f"[*] CTC-aligned samples: {_ok}/{len(train_ds) + len(eval_ds)} (rest use uniform timing)")

# The aligner is no longer needed; free its GPU memory.
del aligner
gc.collect()
torch.cuda.empty_cache()


class VoxtralStreamingCollator:
    """Builds the frame-synchronous Voxtral Realtime batch on the fly.

    Mel features are computed here (not stored in the dataset): `.map()` would turn them into
    huge nested lists in Arrow. input_ids/labels are built per sample with `build_stream`.
    """

    def __init__(self):
        self.leading_space = True  # tokenize non-first words with a leading space (see selection below)
        self.truncated_total = 0
        self._cache = {}
        self._prompt_checked = False

    def _word_ids(self, word, first):
        use_space = self.leading_space and not first
        key = (word, use_space)
        ids = self._cache.get(key)
        if ids is None:
            ids = tokenizer((" " + word) if use_space else word, add_special_tokens=False)["input_ids"]
            ids = [int(i) for i in ids]
            self._cache[key] = ids
        return ids

    def __call__(self, samples):
        feats, rows = [], []
        for s in samples:
            wav, sr = _extract_audio_array(s)
            enc = processor(audio=np.asarray(wav, dtype=np.float32), sampling_rate=sr, return_tensors="pt")
            if not self._prompt_checked:
                if enc["input_ids"][0].tolist() != prompt_ids:
                    raise RuntimeError("processor prompt differs between samples; layout assumption broken")
                self._prompt_checked = True
            f = enc["input_features"][0]  # (n_mels, T)
            n_steps = math.ceil(f.shape[-1] / alen)

            words = list(s["words"])
            word_ids = [self._word_ids(w, i == 0) for i, w in enumerate(words)]
            end_steps = [n_left + int(e / frame_sec) for e in s["word_end_sec"]]
            inp, lab, trunc = build_stream(word_ids, end_steps, n_steps, prompt_ids, PAD_ID, W_ID, n_delay)
            self.truncated_total += trunc
            feats.append(f)
            rows.append((inp, lab, n_steps))

        t_max = max(f.shape[-1] for f in feats)
        t_max = int(math.ceil(t_max / alen) * alen)
        n_max = t_max // alen  # == number of audio embeddings the model will produce
        B = len(rows)
        input_ids = torch.full((B, n_max), PAD_ID, dtype=torch.long)
        attention_mask = torch.zeros((B, n_max), dtype=torch.long)
        labels = torch.full((B, n_max), -100, dtype=torch.long)
        input_features = torch.zeros((B, feats[0].shape[0], t_max), dtype=torch.float32)
        for i, (inp, lab, n) in enumerate(rows):
            input_ids[i, :n] = torch.tensor(inp, dtype=torch.long)
            attention_mask[i, :n] = 1
            labels[i, :n] = torch.tensor(lab, dtype=torch.long)
            input_features[i, :, : feats[i].shape[-1]] = feats[i]
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "input_features": input_features.to(torch.bfloat16),
            "labels": labels,
            # must be a plain int: the model does torch.full((1,), num_delay_tokens)
            "num_delay_tokens": n_delay,
        }


collator = VoxtralStreamingCollator()
pin = use_cuda and input_device.type == "cuda"
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, collate_fn=collator, pin_memory=pin)
eval_loader = DataLoader(eval_ds, batch_size=2, shuffle=False, collate_fn=collator, pin_memory=pin)

# =====================================================================
# 4. TRAINING & AUDIT LOOP
# =====================================================================
epochs = 3
grad_accum = 16
max_grad_norm = 1.0
steps_per_epoch = max(1, math.ceil(len(train_loader) / grad_accum))  # includes a final partial-accumulation step
total_steps = steps_per_epoch * epochs
warmup_steps = max(1, int(0.1 * total_steps))
log_every = max(1, min(20, total_steps))
eval_every = max(1, min(100, total_steps))
print(f"[*] total optimizer steps={total_steps} | warmup={warmup_steps} | log_every={log_every} | eval_every={eval_every}")

scheduler = get_cosine_schedule_with_warmup(
    topo_optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)


def _move_batch_to_device(batch):
    moved = {}
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            moved[k] = v.to(input_device, non_blocking=pin)
        else:
            moved[k] = v
    return moved


def forward_loss(batch):
    """Cross-entropy against the frame-synchronous targets.

    labels[:, j] is already the target for output position j (the model emits one token per
    audio step), so there is deliberately NO label shift here.
    """
    labels = batch["labels"]
    out = model(**{k: v for k, v in batch.items() if k != "labels"})
    logits = out.logits
    if not (labels != -100).any():
        return logits.sum() * 0.0
    return F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), labels.reshape(-1), ignore_index=-100)


@torch.no_grad()
def evaluate_loss():
    was_training = model.training
    model.eval()
    losses = []
    for batch in eval_loader:
        batch = _move_batch_to_device(batch)
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            losses.append(forward_loss(batch).item())
    if was_training:
        model.train()
    return sum(losses) / max(len(losses), 1)


def save_trainable(path):
    state = {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}
    torch.save(state, os.path.join(path, "trainable_weights.pt"))


# ---- layout self-check: show exactly what the model will be trained on ----
def _show_layout_probe():
    s = eval_ds[0]
    b = collator([s])
    n = int(b["attention_mask"][0].sum())
    lab = b["labels"][0, :n].tolist()
    nonpad = [t for t in lab if t not in (-100, PAD_ID)]
    text_ids = [t for t in nonpad if t != W_ID]
    try:
        decoded = tokenizer.decode(text_ids, skip_special_tokens=True)
    except Exception as e:
        decoded = f"<decode failed: {e!r}>"
    print(
        "[probe] batch shapes:",
        {k: (tuple(v.shape) if isinstance(v, torch.Tensor) else v) for k, v in b.items()},
    )
    print(f"[probe] steps={n} | non-pad targets={len(nonpad)} ({100.0 * len(nonpad) / max(1, n - P + 1):.1f}% of predicted steps)")
    print(f"[probe] words: {list(zip(s['words'][:4], [round(x, 2) for x in s['word_end_sec'][:4]]))} ...")
    print(f"[probe] target text : {decoded!r}")
    print(f"[probe] transcript  : {' '.join(s['words'])!r}")
    a, b2 = tokenizer(" hello", add_special_tokens=False)["input_ids"], tokenizer("hello", add_special_tokens=False)["input_ids"]
    print(f"[probe] ' hello' -> {a} | 'hello' -> {b2}  (identical => leading-space choice is moot)")


_show_layout_probe()

# Choose between "' word'" and "'word'" subword tokenization by baseline loss (the checkpoint
# already knows its own format, so the right choice gives clearly lower loss).
_ls_mode = os.environ.get("VOXTRAL_LEADING_SPACE", "auto")
if _ls_mode in ("0", "1"):
    collator.leading_space = _ls_mode == "1"
else:
    _res = {}
    for _flag in (True, False):
        collator.leading_space = _flag
        _res[_flag] = evaluate_loss()
    collator.leading_space = _res[True] <= _res[False]
    print(f"[layout] baseline val loss: leading_space=True -> {_res[True]:.4f} | False -> {_res[False]:.4f} | using {collator.leading_space}")

print(f"[baseline] validation loss before training: {evaluate_loss():.4f}", flush=True)
print(
    "[baseline] truncated words so far: "
    f"{collator.truncated_total} (should be ~0). A sane baseline is small (roughly < 1-2); "
    "a value near 10+ means the layout is still wrong.",
    flush=True,
)

# Drive must be mounted first in Colab: from google.colab import drive; drive.mount("/content/drive")
_drive_dir = "/content/drive/MyDrive/topogpt/voxtral-mini-4b-realtime-2602-topo-4bit-triton-train"
if os.path.ismount("/content/drive"):
    output_dir = _drive_dir
else:
    output_dir = "/content/voxtral_output"
    print(f"[warn] /content/drive is not mounted; saving to ephemeral {output_dir} instead.")
os.makedirs(output_dir, exist_ok=True)

model.train()
topo_optimizer.zero_grad()
global_step = 0
running_step_loss = 0.0
accum_loss = 0.0


def optimizer_step():
    global global_step, running_step_loss, accum_loss
    torch.nn.utils.clip_grad_norm_(trainable_params, max_grad_norm)
    topo_optimizer.step()
    scheduler.step()
    topo_optimizer.zero_grad()

    global_step += 1
    running_step_loss += accum_loss
    accum_loss = 0.0

    if global_step % log_every == 0:
        avg_loss = running_step_loss / log_every
        lr_now = scheduler.get_last_lr()[0]
        print(f"step {global_step:>5} | train loss {avg_loss:.4f} | lr {lr_now:.2e}", flush=True)
        running_step_loss = 0.0

    if global_step % eval_every == 0:
        print(f"[eval] step {global_step:>5} | val loss {evaluate_loss():.4f}", flush=True)


for epoch in range(1, epochs + 1):
    micro_step = 0  # reset per epoch so leftover micro-batches are flushed, not carried over
    for batch in train_loader:
        batch = _move_batch_to_device(batch)
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            step_loss = forward_loss(batch)
            loss = step_loss / grad_accum

        loss.backward()
        accum_loss += step_loss.detach().item() / grad_accum
        micro_step += 1

        if micro_step % grad_accum == 0:
            optimizer_step()

    if micro_step % grad_accum != 0:
        # Flush the final partial accumulation window instead of silently dropping it.
        optimizer_step()

    epoch_dir = os.path.join(output_dir, f"epoch-{epoch}")
    os.makedirs(epoch_dir, exist_ok=True)
    save_trainable(epoch_dir)
    print(f"[*] Saved epoch {epoch} trainable weights -> {epoch_dir}", flush=True)

print(f"[final] validation loss: {evaluate_loss():.4f}", flush=True)

max_drift, current_hash, hash_matched = topo_optimizer.audit_telemetry()
if max_drift is None:
    print("[TopoAudit] anchors inactive (embedding is frozen), nothing to audit.")
else:
    print(f"[TopoAudit] Max Anchor Drift: {max_drift:.6e} | Hash Matched: {hash_matched} | Digest: {current_hash[:16]}")

save_trainable(output_dir)
processor.save_pretrained(output_dir)
print("[*] Done.")

[*] transformers 5.22.1 | model_type=voxtral_realtime


Loading weights:   0%|          | 0/711 [00:00<?, ?it/s]

[check] Linear4bit modules: total=234, inside audio_tower=0 (expect 0)
[*] Trainable parameters: 402,899,200 (train_audio_tower_and_adapter=False)
[TopoAdamW] Triton=True | anchors_active=True | digest 4f7baea21466ca28
[*] Input target device: cuda:0
[*] stream layout: prompt_len=39 (n_left=32, n_delay=6) | step=80 ms | PAD_ID=32 | WORD_ID=33
[*] loading CTC aligner facebook/wav2vec2-base-960h ...


Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

Map:   0%|          | 0/65 [00:00<?, ? examples/s]

Filter:   0%|          | 0/65 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Filter:   0%|          | 0/8 [00:00<?, ? examples/s]

[*] train samples: 65 | eval samples: 8
[*] CTC-aligned samples: 73/73 (rest use uniform timing)
[*] total optimizer steps=9 | warmup=1 | log_every=9 | eval_every=9
[probe] batch shapes: {'input_ids': (1, 117), 'attention_mask': (1, 117), 'input_features': (1, 128, 936), 'labels': (1, 117), 'num_delay_tokens': 6}
[probe] steps=117 | non-pad targets=18 (22.8% of predicted steps)
[probe] words: [('Only', 0.92), ('unfortunately', 1.96), ('his', 2.52), ('own', 2.8)] ...
[probe] target text : 'Only unfortunately his own work never does get good'
[probe] transcript  : 'Only unfortunately his own work never does get good'
[probe] ' hello' -> [52528] | 'hello' -> [29706]  (identical => leading-space choice is moot)
[layout] baseline val loss: leading_space=True -> 0.4510 | False -> 0.4945 | using True
[baseline] validation loss before training: 0.4510
[baseline] truncated words so far: 0 (should be ~0). A sane baseline is small (roughly < 1-2); a value near 10+ means the layout is still wrong.

new

In [ ]:
import os
import logging
import warnings

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("HF_HUB_CACHE", "/content/hf_cache")
logging.getLogger("huggingface_hub.utils._http").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning, message=r".*rope_scaling.*")
warnings.filterwarnings("ignore", message=r".*large amount of egress.*")
warnings.filterwarnings("ignore", message=r".*inner dimension.*not aligned.*")

import gc
import math
import re
import uuid
import hashlib
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch.optim import Optimizer
from torch.utils.data import DataLoader
import transformers
from packaging.version import Version
from transformers import (
    AutoConfig,
    AutoProcessor,
    BitsAndBytesConfig,
    VoxtralRealtimeForConditionalGeneration,
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
    get_cosine_schedule_with_warmup,
)
from transformers.utils import logging as hf_logging
from datasets import Audio, load_dataset

hf_logging.set_verbosity_error()
logging.getLogger("huggingface_hub.utils._http").setLevel(logging.ERROR)

try:
    import triton
    import triton.language as tl

    HAS_TRITON = True
except ImportError:
    triton = None
    tl = None
    HAS_TRITON = False

# =====================================================================
# 0. SEED & DEVICE
# =====================================================================
SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
use_cuda = num_gpus > 0
if not use_cuda:
    raise RuntimeError("4-bit bitsandbytes loading requires a CUDA GPU.")
primary_device = torch.device("cuda:0")

# =====================================================================
# 1. OPTIMIZER: TopoAdamW with bf16 states, stochastic rounding and
#    error-compensated bf16 parameter updates (fused Triton kernel +
#    PyTorch fallback). Optional frozen "anchor" rows in the embedding.
#    NOTE: the anchor audit compares weights against a clone taken from
#    the same weights, so it is a freeze check, not an external certificate.
# =====================================================================
PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]


def _stochastic_round_(src_fp32: torch.Tensor, dst_bf16: torch.Tensor) -> None:
    """Round fp32 -> bf16 stochastically. NOTE: src_fp32 is modified in place."""
    bits = src_fp32.view(torch.int32)
    bits.add_(torch.randint_like(bits, 0, 1 << 16))
    bits.bitwise_and_(-65536)
    dst_bf16.copy_(src_fp32)


if HAS_TRITON:

    @triton.jit
    def _topo_adamw_fused_step_kernel(
        p_ptr,
        grad_ptr,
        m_buf_ptr,
        v_buf_ptr,
        comp_ptr,
        n_elements,
        lr,
        beta1,
        beta2,
        eps,
        wd,
        bias_corr1,
        bias_corr2,
        seed_m,
        seed_v,
        P_IS_BF16: tl.constexpr,
        BLOCK_SIZE: tl.constexpr,
    ):
        pid = tl.program_id(axis=0)
        offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
        mask = offsets < n_elements

        p_raw = tl.load(p_ptr + offsets, mask=mask, other=0.0)
        g = tl.load(grad_ptr + offsets, mask=mask, other=0.0).to(tl.float32)
        m_prev = tl.load(m_buf_ptr + offsets, mask=mask, other=0.0).to(tl.float32)
        v_prev = tl.load(v_buf_ptr + offsets, mask=mask, other=0.0).to(tl.float32)
        comp_prev = tl.load(comp_ptr + offsets, mask=mask, other=0.0).to(tl.float32)

        w = p_raw.to(tl.float32)

        m = beta1 * m_prev + (1.0 - beta1) * g
        v = beta2 * v_prev + (1.0 - beta2) * g * g

        m_hat = m / bias_corr1
        v_hat = v / bias_corr2
        denom = tl.sqrt(v_hat) + eps
        step_update = -lr * (m_hat / denom + wd * w) + comp_prev

        total = w + step_update
        if P_IS_BF16:
            new_p = total.to(tl.bfloat16)
            new_comp = total - new_p.to(tl.float32)
            tl.store(p_ptr + offsets, new_p, mask=mask)
            tl.store(comp_ptr + offsets, new_comp.to(tl.bfloat16), mask=mask)
        else:
            tl.store(p_ptr + offsets, total, mask=mask)
            tl.store(comp_ptr + offsets, tl.zeros_like(total).to(tl.bfloat16), mask=mask)

        m_bits = m.to(tl.int32, bitcast=True)
        m_jitter = (tl.randint(seed_m, offsets) & 0xFFFF).to(tl.int32)
        m_rounded_bits = (m_bits + m_jitter) & -65536
        m_rounded = m_rounded_bits.to(tl.float32, bitcast=True).to(tl.bfloat16)

        v_bits = v.to(tl.int32, bitcast=True)
        v_jitter = (tl.randint(seed_v, offsets) & 0xFFFF).to(tl.int32)
        v_rounded_bits = (v_bits + v_jitter) & -65536
        v_rounded = v_rounded_bits.to(tl.float32, bitcast=True).to(tl.bfloat16)

        tl.store(m_buf_ptr + offsets, m_rounded, mask=mask)
        tl.store(v_buf_ptr + offsets, v_rounded, mask=mask)


def _triton_param_step(
    p,
    grad,
    m_buf,
    v_buf,
    comp,
    lr,
    beta1,
    beta2,
    eps,
    wd,
    step_t,
    base_seed=SEED,
    param_id=0,
    block_size=1024,
) -> None:
    n_elements = p.numel()
    if n_elements == 0:
        return

    p_c = p.contiguous().view(-1)
    g_c = grad.contiguous().view(-1)
    m_c = m_buf.contiguous().view(-1)
    v_c = v_buf.contiguous().view(-1)
    c_c = comp.contiguous().view(-1)

    bias_corr1 = 1.0 - (beta1**step_t)
    bias_corr2 = 1.0 - (beta2**step_t)
    seed_m = int((base_seed * 1_000_003 + step_t * 97 + param_id * 7919) & 0x7FFFFFFF)
    seed_v = int((base_seed * 1_000_033 + step_t * 193 + param_id * 104729) & 0x7FFFFFFF)

    grid = (triton.cdiv(n_elements, block_size),)
    _topo_adamw_fused_step_kernel[grid](
        p_c,
        g_c,
        m_c,
        v_c,
        c_c,
        n_elements,
        float(lr),
        float(beta1),
        float(beta2),
        float(eps),
        float(wd),
        float(bias_corr1),
        float(bias_corr2),
        seed_m,
        seed_v,
        P_IS_BF16=(p_c.dtype == torch.bfloat16),
        BLOCK_SIZE=block_size,
    )

    if not p.is_contiguous():
        p.copy_(p_c.view_as(p))
    if not m_buf.is_contiguous():
        m_buf.copy_(m_c.view_as(m_buf))
    if not v_buf.is_contiguous():
        v_buf.copy_(v_c.view_as(v_buf))
    if not comp.is_contiguous():
        comp.copy_(c_c.view_as(comp))


def _pytorch_param_step(p, m_buf, v_buf, comp, lr, beta1, beta2, eps, wd, step_t) -> None:
    g = p.grad.float()
    m = m_buf.float().mul_(beta1).add_(g, alpha=1.0 - beta1)
    v = v_buf.float().mul_(beta2).addcmul_(g, g, value=1.0 - beta2)
    del g

    w = p.detach().float().clone()
    denom = (v.sqrt() / math.sqrt(1.0 - beta2**step_t)).add_(eps)
    y = m.div(denom).mul_(-lr / (1.0 - beta1**step_t))
    del denom

    y.add_(w, alpha=-lr * wd)
    if p.dtype == torch.bfloat16:
        y.add_(comp.float())
    _stochastic_round_(m, m_buf)
    _stochastic_round_(v, v_buf)
    del m, v

    total = w.add_(y)
    if p.dtype == torch.bfloat16:
        new = total.to(torch.bfloat16)
        comp.copy_(total.sub_(new.float()))
        p.copy_(new)
    else:
        p.copy_(total.to(p.dtype))
        comp.zero_()
    del y, w, total


class TopoAdamW(Optimizer):
    """AdamW with bf16 moments (stochastic rounding) and compensated bf16 params."""

    def __init__(
        self,
        params,
        lr=1e-4,
        betas=(0.9, 0.999),
        eps=1e-8,
        weight_decay=1e-2,
        embedding_layer=None,
        prime_anchors=None,
        base_seed=SEED,
        use_triton=True,
    ):
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay))
        self.base_seed = base_seed
        self.use_triton = bool(use_triton and HAS_TRITON)
        self.embedding_layer = embedding_layer

        self._param_ids = {}
        for g in self.param_groups:
            for p in g["params"]:
                self._param_ids[id(p)] = len(self._param_ids)

        self.anchors_active = False
        self.valid_anchors = []
        self.baseline = {}
        self.certified_sha256_digest = None

        if embedding_layer is not None and hasattr(embedding_layer, "weight"):
            emb_w = embedding_layer.weight
            optimized = any(p is emb_w for g in self.param_groups for p in g["params"])
            if optimized:
                anchors = list(prime_anchors) if prime_anchors is not None else list(PRIME_ANCHORS)
                self.valid_anchors = [i for i in anchors if i < emb_w.shape[0]]
                self.baseline = {i: emb_w.data[i].detach().clone() for i in self.valid_anchors}
                self.certified_sha256_digest = self._digest()
                self.anchors_active = bool(self.valid_anchors)

        print(
            f"[TopoAdamW] Triton={self.use_triton} | anchors_active={self.anchors_active}"
            + (f" | digest {self.certified_sha256_digest[:16]}" if self.anchors_active else "")
        )

    def _digest(self) -> str:
        w = self.embedding_layer.weight.data
        buf = torch.cat([w[i].flatten().float() for i in self.valid_anchors]).cpu().numpy().tobytes()
        return hashlib.sha256(buf).hexdigest()

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        emb_w = self.embedding_layer.weight if self.anchors_active else None
        if emb_w is not None and emb_w.grad is not None:
            for i in self.valid_anchors:
                emb_w.grad[i].zero_()

        for group in self.param_groups:
            lr, wd, eps = group["lr"], group["weight_decay"], group["eps"]
            beta1, beta2 = group["betas"]
            for p in group["params"]:
                if not p.requires_grad or p.grad is None:
                    continue
                state = self.state[p]
                if not state:
                    state["step"] = 0
                    state["exp_avg"] = torch.zeros_like(p, dtype=torch.bfloat16)
                    state["exp_avg_sq"] = torch.zeros_like(p, dtype=torch.bfloat16)
                    state["comp"] = torch.zeros_like(p, dtype=torch.bfloat16)

                state["step"] += 1
                t = state["step"]
                m_buf, v_buf, comp = state["exp_avg"], state["exp_avg_sq"], state["comp"]
                grad_on_dev = p.grad.to(p.device) if p.grad.device != p.device else p.grad

                if self.use_triton and p.is_cuda and grad_on_dev.is_cuda:
                    _triton_param_step(
                        p,
                        grad_on_dev,
                        m_buf,
                        v_buf,
                        comp,
                        lr,
                        beta1,
                        beta2,
                        eps,
                        wd,
                        t,
                        base_seed=self.base_seed,
                        param_id=self._param_ids[id(p)],
                    )
                else:
                    orig_grad = p.grad
                    p.grad = grad_on_dev
                    try:
                        _pytorch_param_step(p, m_buf, v_buf, comp, lr, beta1, beta2, eps, wd, t)
                    finally:
                        p.grad = orig_grad

                if emb_w is not None and p is emb_w:
                    for i in self.valid_anchors:
                        p[i].copy_(self.baseline[i].to(device=p.device, dtype=p.dtype))
                        comp[i].zero_()

        return loss

    def audit_telemetry(self):
        """Returns (max_drift, digest, matched), or (None, None, None) if anchoring is inactive."""
        if not self.anchors_active:
            return None, None, None
        w = self.embedding_layer.weight.data
        drift = max(
            torch.max(torch.abs(w[i].float() - self.baseline[i].to(w.device).float())).item()
            for i in self.valid_anchors
        )
        h = self._digest()
        return drift, h, h == self.certified_sha256_digest


# =====================================================================
# STREAMING-LAYOUT LOGIC
# =====================================================================
CTC_FRAME_SEC = 0.02
_ALIGN_STRIP = re.compile(r"[^A-Z']")


def split_transcript_words(text):
    display, clean = [], []
    for raw_w in text.split():
        c = _ALIGN_STRIP.sub("", raw_w.upper())
        if c:
            display.append(raw_w)
            clean.append(c)
        elif any(ch.isalnum() for ch in raw_w):
            display.append(raw_w)
            clean.append("")
        elif display:
            display[-1] = display[-1] + raw_w
    return display, clean


def ctc_word_end_frames(emission, clean_words, vocab, blank):
    emission = np.asarray(emission, dtype=np.float64)
    tokens, owner = [], []
    aligned_idx = [i for i, w in enumerate(clean_words) if w]
    if not aligned_idx:
        return None
    for n, i in enumerate(aligned_idx):
        if n > 0:
            tokens.append(vocab["|"])
            owner.append(-1)
        for ch in clean_words[i]:
            tokens.append(vocab[ch])
            owner.append(i)

    T, J = emission.shape[0], len(tokens)
    if J == 0 or T < J:
        return None
    stay = emission[:, blank]
    tok = emission[:, tokens]

    trellis = np.full((T + 1, J + 1), -np.inf, dtype=np.float64)
    trellis[0, 0] = 0.0
    for t in range(T):
        trellis[t + 1, 0] = trellis[t, 0] + stay[t]
        trellis[t + 1, 1:] = np.maximum(trellis[t, 1:] + stay[t], trellis[t, :-1] + tok[t])
    if not np.isfinite(trellis[T, J]):
        return None

    emit = [-1] * J
    t, j = T, J
    while j > 0 and t > 0:
        stayed = trellis[t - 1, j] + stay[t - 1]
        changed = trellis[t - 1, j - 1] + tok[t - 1, j - 1]
        if changed > stayed:
            emit[j - 1] = t - 1
            j -= 1
        t -= 1
    if j > 0:
        return None

    ends = [None] * len(clean_words)
    for k, o in enumerate(owner):
        if o >= 0:
            ends[o] = emit[k] if ends[o] is None else max(ends[o], emit[k])
    return ends


def frames_to_end_secs(end_frames):
    out, last = [], 0.0
    for f in end_frames:
        sec = last if f is None else (f + 1) * CTC_FRAME_SEC
        out.append(sec)
        last = sec
    return out


def uniform_word_ends(words, duration_sec):
    weights = np.array([max(1, len(w)) for w in words], dtype=np.float64)
    return (duration_sec * np.cumsum(weights) / weights.sum()).tolist()


def build_stream(word_ids, end_steps, n_steps, prompt_ids, pad_id, word_id, delay):
    P = len(prompt_ids)
    y = [pad_id] * n_steps
    cursor = 0
    truncated = 0
    for ids, f_end in zip(word_ids, end_steps):
        start = f_end + delay
        if start > cursor:
            if start >= n_steps:
                truncated += 1
                continue
            y[start] = word_id
            pos = start + 1
        else:
            pos = cursor
        for tok in ids:
            if pos >= n_steps:
                truncated += 1
                break
            y[pos] = tok
            pos += 1
        cursor = max(cursor, pos)

    input_ids = ([prompt_ids[0]] + y[: n_steps - 1])[:n_steps]
    labels = list(y)
    for j in range(min(P - 1, n_steps)):
        labels[j] = -100
    return input_ids, labels, truncated


# =====================================================================
# 2. MODEL / PROCESSOR (mistralai/Voxtral-Mini-4B-Realtime-2602)
# =====================================================================
model_id = "mistralai/Voxtral-Mini-4B-Realtime-2602"

config = AutoConfig.from_pretrained(model_id)
config.use_cache = False

processor = AutoProcessor.from_pretrained(model_id)
tokenizer = processor.tokenizer

has_prequant = getattr(config, "quantization_config", None) is not None
model_kwargs = dict(
    config=config,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)
if not has_prequant:
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        llm_int8_skip_modules=[
            "lm_head",
            "model.audio_tower",
            "model.multi_modal_projector",
            "model.time_embedding",
        ],
    )

print(
    f"[*] transformers {transformers.__version__} | "
    f"model_type={getattr(config, 'model_type', 'unknown')}"
)
_dtype_kw = "dtype" if Version(transformers.__version__) >= Version("4.56.0") else "torch_dtype"
model = VoxtralRealtimeForConditionalGeneration.from_pretrained(
    model_id, **{_dtype_kw: torch.bfloat16}, **model_kwargs
)
model.config.use_cache = False
gc.collect()
torch.cuda.empty_cache()

embedding_layer = (
    model.get_input_embeddings() if hasattr(model, "get_input_embeddings") else None
)

train_audio_tower_and_adapter = os.environ.get("TRAIN_AUDIO_TOWER_AND_ADAPTER", "0") == "1"

for p in model.parameters():
    p.requires_grad = False

decay_params, no_decay_params, seen = [], [], set()
for name, mod in model.named_modules():
    is_conv = isinstance(mod, (torch.nn.Conv1d, torch.nn.Conv2d))
    if is_conv:
        continue
    is_head = name.endswith("lm_head")
    is_norm = "norm" in name.lower()
    is_audio_or_proj = train_audio_tower_and_adapter and any(
        tok in name.lower()
        for tok in ("audio", "projector", "adapter", "multi_modal", "cross_attn", "q_proj", "k_proj", "v_proj", "o_proj")
    )
    if not (is_head or is_norm or is_audio_or_proj):
        continue
    for p in mod.parameters(recurse=False):
        if id(p) in seen or not p.is_floating_point():
            continue
        seen.add(id(p))
        p.requires_grad = True
        use_decay = (is_head or is_audio_or_proj) and p.ndim >= 2
        (decay_params if use_decay else no_decay_params).append(p)

trainable_params = decay_params + no_decay_params
if not trainable_params:
    raise RuntimeError("No trainable parameters found; check Voxtral module names.")

if hasattr(model, "gradient_checkpointing_enable"):
    try:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    except Exception:
        model.gradient_checkpointing_enable()

try:
    model.enable_input_require_grads()
except Exception as e:
    print(f"[warn] enable_input_require_grads failed: {e!r}")

print(
    f"[*] Trainable parameters: {sum(p.numel() for p in trainable_params):,} "
    f"(train_audio_tower_and_adapter={train_audio_tower_and_adapter})"
)

WEIGHT_DECAY = 1e-2
topo_optimizer = TopoAdamW(
    [
        {"params": decay_params, "weight_decay": WEIGHT_DECAY},
        {"params": no_decay_params, "weight_decay": 0.0},
    ],
    lr=1e-4,
    embedding_layer=embedding_layer,
    prime_anchors=PRIME_ANCHORS,
    base_seed=SEED,
    use_triton=True,
)


def _get_input_device() -> torch.device:
    emb = embedding_layer
    if emb is not None and hasattr(emb, "weight") and emb.weight.device.type != "meta":
        return emb.weight.device
    return primary_device


input_device = _get_input_device()
print(f"[*] Input target device: {input_device}")

# =====================================================================
# 3. STREAMING LAYOUT CONSTANTS + SCALED SPEECH DATA & COLLATOR
# =====================================================================
_audio_cfg = processor.mistral_common_audio_config
n_left = int(_audio_cfg.n_left_pad_tokens)
n_delay = int(processor.num_delay_tokens)
alen = int(getattr(model.config, "audio_length_per_tok", 8))
hop = int(processor.feature_extractor.hop_length)
frame_sec = hop * alen / 16000.0

_probe_enc = processor(audio=np.zeros(16000, dtype=np.float32), sampling_rate=16000, return_tensors="pt")
prompt_ids = _probe_enc["input_ids"][0].tolist()
P = len(prompt_ids)
PAD_ID = prompt_ids[1]
if not (all(t == PAD_ID for t in prompt_ids[1:]) and P == 1 + n_left + n_delay):
    raise RuntimeError(f"Unexpected processor prompt layout: len={P}, n_left={n_left}, n_delay={n_delay}")

_tek = processor.tokenizer.tokenizer.instruct_tokenizer.tokenizer
if not _tek.is_special("[STREAMING_WORD]"):
    raise RuntimeError("Tokenizer has no [STREAMING_WORD] token; cannot build streaming targets.")
W_ID = int(_tek.get_special_token("[STREAMING_WORD]"))
print(
    f"[*] stream layout: prompt_len={P} (n_left={n_left}, n_delay={n_delay}) | step={frame_sec * 1000:.0f} ms | "
    f"PAD_ID={PAD_ID} | WORD_ID={W_ID}"
)

# Valid splits for openslr/librispeech_asr are: 'test', 'train.100', 'train.360', 'validation'
audio_dataset_id = os.environ.get("VOXTRAL_AUDIO_DATASET", "openslr/librispeech_asr")
audio_config_name = os.environ.get("VOXTRAL_AUDIO_CONFIG", "clean")
audio_split = os.environ.get("VOXTRAL_AUDIO_SPLIT", "train.100")
eval_split = os.environ.get("VOXTRAL_EVAL_SPLIT", "validation")
max_train_samples = int(os.environ.get("VOXTRAL_MAX_TRAIN_SAMPLES", "12500"))
max_eval_samples = int(os.environ.get("VOXTRAL_MAX_EVAL_SAMPLES", "500"))
ALIGNER_ID = os.environ.get("VOXTRAL_ALIGNER", "facebook/wav2vec2-base-960h")
MAX_AUDIO_SEC = float(os.environ.get("VOXTRAL_MAX_AUDIO_SEC", "30"))
normalize_text = os.environ.get("VOXTRAL_NORMALIZE_TEXT", "1") == "1"


def _load_and_prepare_split(dataset_id, config_name, split_name, limit_count, aligner):
    cache_dir = os.path.join("/content/hf_cache", dataset_id.replace("/", "__"))
    os.makedirs(cache_dir, exist_ok=True)
    ds = (
        load_dataset(dataset_id, config_name, split=split_name, cache_dir=cache_dir)
        if config_name
        else load_dataset(dataset_id, split=split_name, cache_dir=cache_dir)
    )

    audio_col = None
    for cand in ("audio", "speech"):
        if cand in ds.column_names:
            audio_col = cand
            break
    if audio_col is None:
        for col, feat in ds.features.items():
            if isinstance(feat, Audio) or "audio" in col.lower() or "speech" in col.lower():
                audio_col = col
                break
    if audio_col is None:
        raise KeyError(f"Could not find an audio column in columns={ds.column_names}")

    ds = ds.cast_column(audio_col, Audio(sampling_rate=16000)).shuffle(seed=SEED)
    n_take = min(len(ds), max(1, limit_count))
    ds = ds.select(range(n_take))

    def _extract_audio(sample):
        audio = sample.get(audio_col)
        if isinstance(audio, dict) and "array" in audio:
            return audio["array"], int(audio.get("sampling_rate") or 16000)
        if hasattr(audio, "get_all_samples"):
            decoded = audio.get_all_samples()
            arr = decoded.data.squeeze(0).detach().cpu().numpy()
            sr = int(getattr(decoded, "sample_rate", 16000))
            return arr, sr
        for arr_attr in ("array", "waveform"):
            if hasattr(audio, arr_attr):
                arr = getattr(audio, arr_attr)
                if hasattr(arr, "detach"):
                    arr = arr.detach().cpu().numpy()
                sr = int(getattr(audio, "sampling_rate", getattr(audio, "sample_rate", 16000)))
                return arr, sr
        raise KeyError(f"Unsupported audio payload type={type(audio).__name__}")

    def _extract_text(sample):
        for col in ("text", "transcription", "sentence", "transcript"):
            val = sample.get(col)
            if isinstance(val, str) and val.strip():
                text = val.strip()
                if normalize_text and text.isupper():
                    text = text.lower()
                    text = text[:1].upper() + text[1:]
                return text
        return ""

    def _add_alignment(sample):
        wav, sr = _extract_audio(sample)
        dur = len(wav) / float(sr)
        display, clean = split_transcript_words(_extract_text(sample))
        if not display:
            return {"words": [], "word_end_sec": [], "align_ok": False, "dur_sec": dur, "_audio_col": audio_col}
        ends = None
        try:
            ends = aligner.word_end_secs(wav, sr, clean)
        except Exception as e:
            print(f"[warn] alignment failed ({e!r}); using uniform timing for this sample")
        ok = ends is not None
        if not ok:
            ends = uniform_word_ends(display, dur)
        return {
            "words": display,
            "word_end_sec": [float(x) for x in ends],
            "align_ok": ok,
            "dur_sec": dur,
            "_audio_col": audio_col,
        }

    mapped = ds.map(_add_alignment, load_from_cache_file=False, new_fingerprint=uuid.uuid4().hex)
    return mapped.filter(
        lambda words, dur: len(words) > 0 and dur <= MAX_AUDIO_SEC,
        input_columns=["words", "dur_sec"],
    )


class CTCWordAligner:
    def __init__(self, aligner_id, device):
        self.device = device
        self.proc = Wav2Vec2Processor.from_pretrained(aligner_id)
        self.model = Wav2Vec2ForCTC.from_pretrained(aligner_id).to(device).eval()
        self.vocab = {k.upper(): v for k, v in self.proc.tokenizer.get_vocab().items()}
        self.blank = self.proc.tokenizer.pad_token_id

    @torch.no_grad()
    def word_end_secs(self, wav, sr, clean_words):
        x = self.proc(np.asarray(wav, dtype=np.float32), sampling_rate=sr, return_tensors="pt").input_values
        logits = self.model(x.to(self.device)).logits[0].float()
        emission = torch.log_softmax(logits, dim=-1).cpu().numpy()
        frames = ctc_word_end_frames(emission, clean_words, self.vocab, self.blank)
        return None if frames is None else frames_to_end_secs(frames)


print(f"[*] loading CTC aligner {ALIGNER_ID} ...")
aligner = CTCWordAligner(ALIGNER_ID, input_device)

train_ds = _load_and_prepare_split(audio_dataset_id, audio_config_name, audio_split, max_train_samples, aligner)
eval_ds = _load_and_prepare_split(audio_dataset_id, audio_config_name, eval_split, max_eval_samples, aligner)

print(f"[*] train samples: {len(train_ds)} | eval samples: {len(eval_ds)}")
_ok = sum(bool(x) for x in train_ds["align_ok"]) + sum(bool(x) for x in eval_ds["align_ok"])
print(f"[*] CTC-aligned samples: {_ok}/{len(train_ds) + len(eval_ds)} (rest use uniform timing)")

del aligner.model, aligner.proc, aligner
gc.collect()
torch.cuda.empty_cache()


class VoxtralStreamingCollator:
    def __init__(self):
        self.leading_space = True
        self.truncated_total = 0
        self._cache = {}
        self._prompt_checked = False

    def _word_ids(self, word, first):
        use_space = self.leading_space and not first
        key = (word, use_space)
        ids = self._cache.get(key)
        if ids is None:
            ids = tokenizer((" " + word) if use_space else word, add_special_tokens=False)["input_ids"]
            ids = [int(i) for i in ids]
            self._cache[key] = ids
        return ids

    def __call__(self, samples):
        feats, rows = [], []
        for s in samples:
            acol = s["_audio_col"]
            audio = s.get(acol)
            if isinstance(audio, dict) and "array" in audio:
                wav, sr = audio["array"], int(audio.get("sampling_rate") or 16000)
            elif hasattr(audio, "get_all_samples"):
                decoded = audio.get_all_samples()
                wav = decoded.data.squeeze(0).detach().cpu().numpy()
                sr = int(getattr(decoded, "sample_rate", 16000))
            else:
                wav = audio["array"]
                sr = int(audio.get("sampling_rate") or 16000)

            enc = processor(audio=np.asarray(wav, dtype=np.float32), sampling_rate=sr, return_tensors="pt")
            if not self._prompt_checked:
                if enc["input_ids"][0].tolist() != prompt_ids:
                    raise RuntimeError("processor prompt differs between samples; layout assumption broken")
                self._prompt_checked = True
            f = enc["input_features"][0]
            n_steps = math.ceil(f.shape[-1] / alen)

            words = list(s["words"])
            word_ids = [self._word_ids(w, i == 0) for i, w in enumerate(words)]
            end_steps = [n_left + int(e / frame_sec) for e in s["word_end_sec"]]
            inp, lab, trunc = build_stream(word_ids, end_steps, n_steps, prompt_ids, PAD_ID, W_ID, n_delay)
            self.truncated_total += trunc
            feats.append(f)
            rows.append((inp, lab, n_steps))

        t_max = max(f.shape[-1] for f in feats)
        t_max = int(math.ceil(t_max / alen) * alen)
        n_max = t_max // alen
        B = len(rows)
        input_ids = torch.full((B, n_max), PAD_ID, dtype=torch.long)
        attention_mask = torch.zeros((B, n_max), dtype=torch.long)
        labels = torch.full((B, n_max), -100, dtype=torch.long)
        input_features = torch.zeros((B, feats[0].shape[0], t_max), dtype=torch.float32)
        for i, (inp, lab, n) in enumerate(rows):
            input_ids[i, :n] = torch.tensor(inp, dtype=torch.long)
            attention_mask[i, :n] = 1
            labels[i, :n] = torch.tensor(lab, dtype=torch.long)
            input_features[i, :, : feats[i].shape[-1]] = feats[i]
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "input_features": input_features.to(torch.bfloat16),
            "labels": labels,
            "num_delay_tokens": n_delay,
        }


collator = VoxtralStreamingCollator()
pin = use_cuda and input_device.type == "cuda"
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, collate_fn=collator, pin_memory=pin)
eval_loader = DataLoader(eval_ds, batch_size=2, shuffle=False, collate_fn=collator, pin_memory=pin)

# =====================================================================
# 4. TRAINING & TOPO-CBP AUDIT LOOP
# =====================================================================
epochs = int(os.environ.get("VOXTRAL_EPOCHS", "3"))
grad_accum = int(os.environ.get("VOXTRAL_GRAD_ACCUM", "16"))
max_grad_norm = 1.0
steps_per_epoch = max(1, math.ceil(len(train_loader) / grad_accum))
total_steps = steps_per_epoch * epochs
warmup_steps = max(1, int(0.1 * total_steps))
log_every = max(1, min(20, total_steps))
eval_every = max(1, min(100, total_steps))
print(f"[*] total optimizer steps={total_steps} | warmup={warmup_steps} | log_every={log_every} | eval_every={eval_every}")

scheduler = get_cosine_schedule_with_warmup(
    topo_optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)


def _move_batch_to_device(batch):
    moved = {}
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            moved[k] = v.to(input_device, non_blocking=pin)
        else:
            moved[k] = v
    return moved


def forward_loss(batch):
    labels = batch["labels"]
    out = model(**{k: v for k, v in batch.items() if k != "labels"})
    logits = out.logits
    if not (labels != -100).any():
        return logits.sum() * 0.0
    return F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), labels.reshape(-1), ignore_index=-100)


@torch.no_grad()
def evaluate_loss(loader=eval_loader):
    was_training = model.training
    model.eval()
    losses = []
    for batch in loader:
        batch = _move_batch_to_device(batch)
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            losses.append(forward_loss(batch).item())
    if was_training:
        model.train()
    return sum(losses) / max(len(losses), 1)


def save_trainable(path):
    state = {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}
    torch.save(state, os.path.join(path, "trainable_weights.pt"))


_ls_mode = os.environ.get("VOXTRAL_LEADING_SPACE", "auto")
if _ls_mode in ("0", "1"):
    collator.leading_space = _ls_mode == "1"
else:
    _res = {}
    for _flag in (True, False):
        collator.leading_space = _flag
        _res[_flag] = evaluate_loss()
    collator.leading_space = _res[True] <= _res[False]
    print(f"[layout] baseline val loss: leading_space=True -> {_res[True]:.4f} | False -> {_res[False]:.4f} | using {collator.leading_space}")

collator.truncated_total = 0
baseline_val_loss = evaluate_loss()
print(f"[baseline] validation loss before training: {baseline_val_loss:.4f}", flush=True)
print(f"[baseline] truncated words in clean eval pass: {collator.truncated_total}", flush=True)
collator.truncated_total = 0

_drive_dir = "/content/drive/MyDrive/topogpt/voxtral-mini-4b-realtime-2602-topo-4bit-triton-train"
if os.path.ismount("/content/drive"):
    output_dir = _drive_dir
else:
    output_dir = "/content/voxtral_output"
    print(f"[warn] /content/drive is not mounted; saving to ephemeral {output_dir} instead.")
os.makedirs(output_dir, exist_ok=True)

model.train()
topo_optimizer.zero_grad()
global_step = 0
running_step_loss = 0.0
accum_loss = 0.0


def optimizer_step():
    global global_step, running_step_loss, accum_loss
    torch.nn.utils.clip_grad_norm_(trainable_params, max_grad_norm)
    topo_optimizer.step()
    scheduler.step()
    topo_optimizer.zero_grad()

    global_step += 1
    running_step_loss += accum_loss
    accum_loss = 0.0

    if global_step % log_every == 0:
        avg_loss = running_step_loss / log_every
        lr_now = scheduler.get_last_lr()[0]
        print(f"step {global_step:>5} | train loss {avg_loss:.4f} | lr {lr_now:.2e}", flush=True)
        running_step_loss = 0.0

    if global_step % eval_every == 0:
        print(f"[eval] step {global_step:>5} | val loss {evaluate_loss():.4f}", flush=True)


for epoch in range(1, epochs + 1):
    micro_step = 0
    for batch in train_loader:
        batch = _move_batch_to_device(batch)
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            step_loss = forward_loss(batch)
            loss = step_loss / grad_accum

        loss.backward()
        accum_loss += step_loss.detach().item() / grad_accum
        micro_step += 1

        if micro_step % grad_accum == 0:
            optimizer_step()

    if micro_step % grad_accum != 0:
        optimizer_step()

    epoch_dir = os.path.join(output_dir, f"epoch-{epoch}")
    os.makedirs(epoch_dir, exist_ok=True)
    save_trainable(epoch_dir)
    print(f"[*] Saved epoch {epoch} trainable weights -> {epoch_dir}", flush=True)

final_val_loss = evaluate_loss()
print(f"[final] validation loss: {final_val_loss:.4f} (delta={final_val_loss - baseline_val_loss:+.4f})", flush=True)

max_drift, current_hash, hash_matched = topo_optimizer.audit_telemetry()
if max_drift is None:
    print("[TopoAudit] anchors inactive (embedding is frozen), nothing to audit.")
else:
    print(f"[TopoAudit] Max Anchor Drift: {max_drift:.6e} | Hash Matched: {hash_matched} | Digest: {current_hash[:16]}")

save_trainable(output_dir)
processor.save_pretrained(output_dir)
print("[*] Done.")

[*] transformers 5.22.1 | model_type=voxtral_realtime


Loading weights:   0%|          | 0/711 [00:00<?, ?it/s]

[*] Trainable parameters: 402,899,200 (train_audio_tower_and_adapter=False)
[TopoAdamW] Triton=True | anchors_active=True | digest 4f7baea21466ca28
[*] Input target device: cuda:0
[*] stream layout: prompt_len=39 (n_left=32, n_delay=6) | step=80 ms | PAD_ID=32 | WORD_ID=33
[*] loading CTC aligner facebook/wav2vec2-base-960h ...


Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

... (3 outputs left out)

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

[*] train samples: 12500 | eval samples: 499
[*] CTC-aligned samples: 12999/12999 (rest use uniform timing)
[*] total optimizer steps=1173 | warmup=117 | log_every=20 | eval_every=100
[layout] baseline val loss: leading_space=True -> 0.4814 | False -> 0.5725 | using True
[baseline] validation loss before training: 0.4814
[baseline] truncated words in clean eval pass: 0
step    20 | train loss 0.3486 | lr 1.71e-05
step    40 | train loss 0.2986 | lr 3.42e-05
step    60 | train loss 0.2438 | lr 5.13e-05
step    80 | train loss 0.1978 | lr 6.84e-05
step   100 | train loss 0.1702 | lr 8.55e-05
[eval] step   100 | val loss 0.1823
step   120 | train loss 0.1572 | lr 1.00e-04
step   140 | train loss 0.1350 | lr 9.99e-05
step   160 | train loss 0.1291 | lr 9.96e-05
step   180 | train loss 0.1211 | lr 9.91e-05
step   200 | train loss 0.1155 | lr 9.85e-05
[eval] step   200 | val loss 0.1292
step   220 | train loss 0.1144 | lr 9.77e-05
step   240 | train loss 0.1081 | lr 9.67e-05
step   260 | tra

In [ ]:
%cd /content/drive/MyDrive/topogpt/voxtral-mini-4b-realtime-2602-topo-4bit-triton-train
!du -sh *

/content/drive/MyDrive/topogpt/voxtral-mini-4b-realtime-2602-topo-4bit-triton-train
769M	epoch-1
769M	epoch-2
769M	epoch-3
512	processor_config.json
15M	tekken.json
769M	trainable_weights.pt


## merger

In [ ]:
import os
import torch
from transformers import AutoProcessor, VoxtralRealtimeForConditionalGeneration

base_model_id = "mistralai/Voxtral-Mini-4B-Realtime-2602"
ckpt_dir = "/content/drive/MyDrive/topogpt/voxtral-mini-4b-realtime-2602-topo-4bit-triton-train"
weights_path = os.path.join(ckpt_dir, "trainable_weights.pt")  # or epoch-3/trainable_weights.pt
merged_dir = "/content/voxtral_merged_hf"

os.makedirs(merged_dir, exist_ok=True)
model = VoxtralRealtimeForConditionalGeneration.from_pretrained(
    base_model_id, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True
)
state = torch.load(weights_path, map_location="cpu")
missing, unexpected = model.load_state_dict(state, strict=False)
print(f"[merge] loaded keys={len(state)} | unexpected={len(unexpected)}")

model.save_pretrained(merged_dir, safe_serialization=True)
processor = AutoProcessor.from_pretrained(base_model_id)
processor.save_pretrained(merged_dir)

Loading weights:   0%|          | 0/711 [00:00<?, ?it/s]

[merge] loaded keys=119 | unexpected=0


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['/content/voxtral_merged_hf/processor_config.json']

In [ ]:
import gc
import hashlib
import numpy as np
import torch
import transformers
from packaging.version import Version
from transformers import AutoProcessor, VoxtralRealtimeForConditionalGeneration

BASE_ID = "mistralai/Voxtral-Mini-4B-Realtime-2602"
MERGED_DIR = "/content/voxtral_merged_hf"
PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]

dtype_kw = "dtype" if Version(transformers.__version__) >= Version("4.56.0") else "torch_dtype"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

base_model = VoxtralRealtimeForConditionalGeneration.from_pretrained(
    BASE_ID, **{dtype_kw: torch.bfloat16}, low_cpu_mem_usage=True
)
merged_model = VoxtralRealtimeForConditionalGeneration.from_pretrained(
    MERGED_DIR, **{dtype_kw: torch.bfloat16}, low_cpu_mem_usage=True
)

base_sd = base_model.state_dict()
merged_sd = merged_model.state_dict()

assert set(base_sd.keys()) == set(merged_sd.keys()), "State dict keys do not match!"

changed_keys = []
max_diff = 0.0
has_non_finite = False

for name, m_tensor in merged_sd.items():
    if not m_tensor.is_floating_point():
        continue
    if not torch.isfinite(m_tensor).all():
        has_non_finite = True
        print(f"[FAIL] Non-finite values in {name}")
    b_tensor = base_sd[name].to(dtype=m_tensor.dtype, device=m_tensor.device)
    diff = (m_tensor.float() - b_tensor.float()).abs().max().item()
    if diff > 0.0:
        changed_keys.append((name, diff))
        if diff > max_diff:
            max_diff = diff

print(f"[parity] total keys={len(merged_sd)} | modified tensors={len(changed_keys)} | max_abs_diff={max_diff:.6e}")
print(f"[parity] all finite={not has_non_finite}")

def _anchor_digest(model):
    emb_w = model.get_input_embeddings().weight.data
    valid = [i for i in PRIME_ANCHORS if i < emb_w.shape[0]]
    buf = torch.cat([emb_w[i].flatten().float() for i in valid]).cpu().numpy().tobytes()
    return hashlib.sha256(buf).hexdigest(), valid

base_hash, valid_anchors = _anchor_digest(base_model)
merged_hash, _ = _anchor_digest(merged_model)
emb_b = base_model.get_input_embeddings().weight.data
emb_m = merged_model.get_input_embeddings().weight.data
anchor_drift = max(
    (emb_m[i].float() - emb_b[i].float()).abs().max().item() for i in valid_anchors
)
print(f"[anchors] drift={anchor_drift:.6e} | base={base_hash[:16]} | merged={merged_hash[:16]} | match={base_hash == merged_hash}")

del base_sd, merged_sd, base_model
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()

processor = AutoProcessor.from_pretrained(MERGED_DIR)
wav = np.zeros(16000, dtype=np.float32)
enc = processor(audio=wav, sampling_rate=16000, return_tensors="pt")
batch = {
    k: (v.to(device) if isinstance(v, torch.Tensor) else v)
    for k, v in enc.items()
}
if "input_features" in batch and batch["input_features"].is_floating_point():
    batch["input_features"] = batch["input_features"].to(torch.bfloat16)

merged_model = merged_model.to(device).eval()
with torch.no_grad():
    if device.type == "cuda":
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            out = merged_model(**batch)
    else:
        out = merged_model(**batch)

logits_finite = torch.isfinite(out.logits).all().item()
print(f"[forward] logits shape={tuple(out.logits.shape)} | finite={logits_finite}")

Loading weights:   0%|          | 0/711 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/711 [00:00<?, ?it/s]

[parity] total keys=712 | modified tensors=120 | max_abs_diff=5.468750e-02
[parity] all finite=True
[anchors] drift=0.000000e+00 | base=4f7baea21466ca28 | merged=4f7baea21466ca28 | match=True


RuntimeError: The size of tensor a (39) must match the size of tensor b (62) at non-singleton dimension 1

## gguf-convertor

In [ ]:
!ls -ltha /content/drive/MyDrive/topogpt/gguf/

!ls -ltha /content/voxtral_merged_hf

total 12K
drwx------ 5 root root 4.0K Nov 24 01:12 ..
drwx------ 2 root root 4.0K Nov 21 04:59 nemotron
drwx------ 3 root root 4.0K Oct 26 15:10 .
total 8.3G
drwxr-xr-x 2 root root 4.0K Nov 24 06:57 .
-rw-r--r-- 1 root root  384 Nov 24 06:57 processor_config.json
-r--r--r-- 1 root root  15M Nov 24 06:57 tekken.json
-rw------- 1 root root 8.3G Nov 24 06:57 model.safetensors
-rw-r--r-- 1 root root 1.7K Nov 24 06:56 config.json
-rw-r--r-- 1 root root  187 Nov 24 06:56 generation_config.json
drwxr-xr-x 1 root root 4.0K Nov 24 06:56 ..


In [ ]:
!ls -ltha /content/drive/MyDrive/topogpt/llama.cpp

total 420K
drwx------ 31 root root 4.0K Nov 24 01:12 ..
drwx------ 31 root root 4.0K Nov  2 11:04 .
drwx------ 14 root root 4.0K Nov  2 11:01 build
drwx------  2 root root 4.0K Nov  2 10:54 .git
drwx------  3 root root 4.0K Nov  2 10:54 models
-rw-------  1 root root  163 Nov  2 10:54 mypy.ini
drwx------  3 root root 4.0K Nov  2 10:54 pocs
-rw-------  1 root root 2.0K Nov  2 10:54 pyproject.toml
-rw-------  1 root root  696 Nov  2 10:54 pyrightconfig.json
drwx------  2 root root 4.0K Nov  2 10:54 requirements
-rw-------  1 root root  551 Nov  2 10:54 requirements.txt
drwx------  6 root root 4.0K Nov  2 10:54 scripts
drwx------  6 root root 4.0K Nov  2 10:54 skills
drwx------  3 root root 4.0K Nov  2 10:54 src
drwx------  6 root root 4.0K Nov  2 10:54 tests
drwx------ 22 root root 4.0K Nov  2 10:54 tools
-rw-------  1 root root  643 Nov  2 10:54 ty.toml
drwx------  8 root root 4.0K Nov  2 10:54 vendor
-rw-------  1 root root  12K Nov  2 10:54 AGENTS.md
drwx------  2 root root 4.0K Nov  

In [ ]:
import os
import subprocess

LLAMA_CPP_DIR = "/content/drive/MyDrive/topogpt/llama.cpp"
MERGED = "/content/voxtral_merged_hf"
GGUF_OUT_DIR = "/content/drive/MyDrive/topogpt/gguf"
GGUF_BF16 = os.path.join(GGUF_OUT_DIR, "voxtral-mini-4b-realtime-2602-topo-triton-bf16.gguf")

if not os.path.isdir(MERGED):
    raise FileNotFoundError(f"Merged HF directory not found at {MERGED}. Run the merge step first.")

os.makedirs(GGUF_OUT_DIR, exist_ok=True)

cmd = [
    "python",
    f"{LLAMA_CPP_DIR}/convert_hf_to_gguf.py",
    MERGED,
    "--outfile",
    GGUF_BF16,
    "--outtype",
    "bf16",
]

proc = subprocess.run(cmd, capture_output=True, text=True)
if proc.returncode != 0:
    print("STDOUT:\n", proc.stdout[-4000:])
    print("STDERR:\n", proc.stderr[-4000:])
    raise RuntimeError(f"convert_hf_to_gguf.py failed with exit code {proc.returncode}")

print(GGUF_BF16, f"{os.path.getsize(GGUF_BF16) / 1e9:.1f} GB")

/content/drive/MyDrive/topogpt/gguf/voxtral-mini-4b-realtime-2602-topo-triton-bf16.gguf 6.9 GB


## gguf-quantize

In [ ]:
!chmod a+x /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize
!ls -ltha /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize

-rwx------ 1 root root 18K Nov  2 10:56 /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize


In [ ]:
import os
import subprocess
from google.colab import drive

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

LLAMA_CPP_DIR = "/content/drive/MyDrive/topogpt/llama.cpp"
GGUF_DIR = "/content/drive/MyDrive/topogpt/gguf"
BF16_IN = os.path.join(GGUF_DIR, "voxtral-mini-4b-realtime-2602-topo-triton-bf16.gguf")
Q8_OUT = os.path.join(GGUF_DIR, "voxtral-mini-4b-realtime-2602-topo-triton-Q8_0.gguf")

if not os.path.isfile(BF16_IN):
    raise FileNotFoundError(f"BF16 GGUF not found at {BF16_IN}. Run convert_hf_to_gguf.py first.")

quantize_candidates = [
    os.path.join(LLAMA_CPP_DIR, "build", "bin", "llama-quantize"),
    os.path.join(LLAMA_CPP_DIR, "llama-quantize"),
]
QUANTIZE_BIN = next((p for p in quantize_candidates if os.path.isfile(p)), quantize_candidates[0])
if not os.path.isfile(QUANTIZE_BIN):
    raise FileNotFoundError(f"llama-quantize binary not found at {QUANTIZE_BIN}. Build llama.cpp first.")

lib_dirs = [
    os.path.join(LLAMA_CPP_DIR, "build", "bin"),
    os.path.join(LLAMA_CPP_DIR, "build", "lib"),
    os.path.join(LLAMA_CPP_DIR, "build"),
]
existing_lib_dirs = [d for d in lib_dirs if os.path.isdir(d)]

env = os.environ.copy()
prev_ld = env.get("LD_LIBRARY_PATH", "")
env["LD_LIBRARY_PATH"] = ":".join(existing_lib_dirs + ([prev_ld] if prev_ld else []))

subprocess.run([QUANTIZE_BIN, BF16_IN, Q8_OUT, "Q8_0"], env=env, check=True)

print(f"[OK] Q8_0 GGUF created: {Q8_OUT} ({os.path.getsize(Q8_OUT) / (1024**3):.2f} GiB)")

[OK] Q8_0 GGUF created: /content/drive/MyDrive/topogpt/gguf/voxtral-mini-4b-realtime-2602-topo-triton-Q8_0.gguf (3.41 GiB)


## gguf-evaluation

In [ ]:
import array
import os
import subprocess
import urllib.request
import wave

WAV_URL = (
    "https://upload.wikimedia.org/wikipedia/commons/3/38/"
    "President_Barack_Obama_Recites_President_Abraham_Lincoln%27s_Gettysburg_Address.wav"
)
RAW_WAV = "/content/gettysburg_raw.wav"
OUT_WAV = "/content/better_speech_16k.wav"

req = urllib.request.Request(
    WAV_URL,
    headers={"User-Agent": "SOMALA-Voxtral-Eval/1.0 (frank.morales@sovereign-machine-lab.ai)"},
)
with urllib.request.urlopen(req, timeout=60) as resp:
    data = resp.read()

with open(RAW_WAV, "wb") as f:
    f.write(data)

subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-i",
        RAW_WAV,
        "-ac",
        "1",
        "-ar",
        "16000",
        "-c:a",
        "pcm_s16le",
        OUT_WAV,
    ],
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

with wave.open(OUT_WAV, "rb") as wf:
    sr = wf.getframerate()
    n_frames = wf.getnframes()
    samples = array.array("h", wf.readframes(n_frames))
    peak = max((abs(x) for x in samples), default=0)

print(f"Saved: {OUT_WAV} | sr={sr} Hz | duration={n_frames / float(sr):.2f}s | peak_int16={peak}")

Saved: /content/better_speech_16k.wav | sr=16000 Hz | duration=105.50s | peak_int16=32768


In [ ]:
import array
import hashlib
import os
import wave
from llama_cpp import Llama

GGUF_Q8 = "/content/drive/MyDrive/topogpt/gguf/voxtral-mini-4b-realtime-2602-topo-triton-Q8_0.gguf"
WAV_PATH = "/content/better_speech_16k.wav"

with wave.open(WAV_PATH, "rb") as wf:
    sr = wf.getframerate()
    n_frames = wf.getnframes()
    pcm = array.array("h", wf.readframes(n_frames))
    peak = max((abs(x) for x in pcm), default=0)

dur_s = n_frames / float(sr)
steps_80ms = int((dur_s * 1000.0) // 80)

llm = Llama(
    model_path=GGUF_Q8,
    n_ctx=1024,
    n_gpu_layers=-1,
    verbose=False,
)

# Hash first 16 MB of the GGUF file for a fast artifact check
with open(GGUF_Q8, "rb") as f:
    head_digest = hashlib.sha256(f.read(16 * 1024 * 1024)).hexdigest()[:16]

print(f"GGUF: {os.path.basename(GGUF_Q8)} | size={os.path.getsize(GGUF_Q8) / 1e9:.2f} GB | head16MB={head_digest}")
print(f"Arch: {(llm.metadata or {}).get('general.architecture')} | vocab={llm.n_vocab()} | bos={llm.token_bos()} eos={llm.token_eos()}")
print(f"Audio: {WAV_PATH} | {dur_s:.2f}s @ {sr} Hz | peak={peak} | ~{steps_80ms} x 80ms frames")

GGUF: voxtral-mini-4b-realtime-2602-topo-triton-Q8_0.gguf | size=3.66 GB | head16MB=d0b60b8e2132e2f3
Arch: mistral3 | vocab=131072 | bos=1 eos=2
Audio: /content/better_speech_16k.wav | 105.50s @ 16000 Hz | peak=32768 | ~1318 x 80ms frames


In [ ]:
import array
import gc
import os
import wave
import numpy as np
import torch
import transformers
from packaging.version import Version
from transformers import AutoProcessor, VoxtralRealtimeForConditionalGeneration

WAV_PATH = "/content/better_speech_16k.wav"
MERGED_DIR = "/content/voxtral_merged_hf"
BASE_MODEL_ID = "mistralai/Voxtral-Mini-4B-Realtime-2602"
MAX_SEC = 30.0

if not os.path.isfile(WAV_PATH):
    raise FileNotFoundError(f"Audio file not found: {WAV_PATH}")

with wave.open(WAV_PATH, "rb") as wf:
    sr = wf.getframerate()
    n_channels = wf.getnchannels()
    sampwidth = wf.getsampwidth()
    n_frames = min(wf.getnframes(), int(round(MAX_SEC * sr)))
    raw = wf.readframes(n_frames)

if sampwidth != 2:
    raise ValueError(f"Expected 16-bit PCM WAV, got sampwidth={sampwidth}")

raw_np = np.frombuffer(raw, dtype="<i2")
if n_channels > 1:
    mono_f32 = raw_np.reshape(-1, n_channels).mean(axis=1).astype(np.float32)
else:
    mono_f32 = raw_np.astype(np.float32)

peak = float(np.max(np.abs(mono_f32))) if mono_f32.size else 1.0
scale = 32768.0 if peak <= 32768.0 else peak
wav_np = np.ascontiguousarray(mono_f32 / (scale if scale > 0 else 1.0), dtype=np.float32)

model_dir = MERGED_DIR if os.path.isdir(MERGED_DIR) else BASE_MODEL_ID
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
dtype_kw = "dtype" if Version(transformers.__version__) >= Version("4.56.0") else "torch_dtype"

print(f"Using model source: {model_dir}")
print(f"Audio array: shape={wav_np.shape}, dtype={wav_np.dtype}, dur={len(wav_np) / sr:.2f}s @ {sr} Hz")

processor = AutoProcessor.from_pretrained(model_dir)
model = (
    VoxtralRealtimeForConditionalGeneration.from_pretrained(
        model_dir,
        **{dtype_kw: torch.bfloat16},
        low_cpu_mem_usage=True,
    )
    .to(device)
    .eval()
)

inputs = processor(audio=wav_np, sampling_rate=sr, return_tensors="pt")
model_inputs = {}
for k, v in inputs.items():
    if isinstance(v, torch.Tensor):
        t = v.to(device)
        if t.is_floating_point():
            t = t.to(torch.bfloat16)
        model_inputs[k] = t
    else:
        model_inputs[k] = v

alen = int(getattr(model.config, "audio_length_per_tok", 8))
n_feat = model_inputs["input_features"].shape[-1] if "input_features" in model_inputs else 0
max_new_tokens = min(1024, max(256, (n_feat // max(1, alen)) + 64))

with torch.no_grad():
    if device.type == "cuda":
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            out_ids = model.generate(**model_inputs, max_new_tokens=max_new_tokens, do_sample=False)
    else:
        out_ids = model.generate(**model_inputs, max_new_tokens=max_new_tokens, do_sample=False)

prompt_len = model_inputs["input_ids"].shape[1] if "input_ids" in model_inputs else 0
gen_ids = out_ids[:, prompt_len:] if (prompt_len and out_ids.shape[1] > prompt_len) else out_ids
transcript = processor.batch_decode(gen_ids, skip_special_tokens=True)[0].strip()

print("\n=== TRANSCRIPTION ===")
print(transcript)

del model, model_inputs, out_ids
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()

Using model source: /content/voxtral_merged_hf
Audio array: shape=(480000,), dtype=float32, dur=30.00s @ 16000 Hz


Loading weights:   0%|          | 0/711 [00:00<?, ?it/s]


=== TRANSCRIPTION ===
Four score and seven years ago our fathers brought forth upon this continent a new nation conceived in liberty and dedicated to the proposition that all men are created equal now we are engaged in a great civil war testing whether that nation or any nation so conceived and so dedicated can long endure we are met on a great battlefield of that war we have come to dedicate a portion of it


In [ ]:
import array
import hashlib
import json
import os
import wave
from llama_cpp import Llama

GGUF_Q8 = "/content/drive/MyDrive/topogpt/gguf/voxtral-mini-4b-realtime-2602-topo-triton-Q8_0.gguf"
WAV_PATH = "/content/better_speech_16k.wav"
REPORT_PATH = "/content/voxtral_q8_preupload_check.json"

if not os.path.isfile(GGUF_Q8):
    raise FileNotFoundError(f"GGUF_Q8 not found: {GGUF_Q8}")
if not os.path.isfile(WAV_PATH):
    raise FileNotFoundError(f"WAV_PATH not found: {WAV_PATH}")

with open(GGUF_Q8, "rb") as f:
    head16 = f.read(16 * 1024 * 1024)
    head16_sha256 = hashlib.sha256(head16).hexdigest()

with wave.open(WAV_PATH, "rb") as wf:
    sr = wf.getframerate()
    n_frames = wf.getnframes()
    pcm = array.array("h", wf.readframes(n_frames))
    peak = max((abs(x) for x in pcm), default=0)

llm = Llama(
    model_path=GGUF_Q8,
    n_ctx=1024,
    n_gpu_layers=-1,
    verbose=False,
)

meta = getattr(llm, "metadata", {}) or {}
report = {
    "gguf_path": GGUF_Q8,
    "gguf_file": os.path.basename(GGUF_Q8),
    "size_gb": round(os.path.getsize(GGUF_Q8) / 1e9, 4),
    "head16mb_sha256": head16_sha256,
    "head16mb_prefix": head16_sha256[:16],
    "arch": meta.get("general.architecture"),
    "vocab_size": llm.n_vocab(),
    "bos_token": llm.token_bos(),
    "eos_token": llm.token_eos(),
    "reference_audio": {
        "path": WAV_PATH,
        "sample_rate": sr,
        "duration_s": round(n_frames / float(sr), 2),
        "peak_int16": peak,
    },
    "upload_role": "quantized_q8_0_gguf_artifact",
    "transcription_path_note": "Use VoxtralRealtimeForConditionalGeneration on HF/merged weights or an audio-capable realtime runtime for WAV transcription; text-only Llama() does not encode audio.",
}

with open(REPORT_PATH, "w", encoding="utf-8") as out:
    json.dump(report, out, indent=2)

print(json.dumps(report, indent=2))
print("Saved pre-upload check:", REPORT_PATH)

{
  "gguf_path": "/content/drive/MyDrive/topogpt/gguf/voxtral-mini-4b-realtime-2602-topo-triton-Q8_0.gguf",
  "gguf_file": "voxtral-mini-4b-realtime-2602-topo-triton-Q8_0.gguf",
  "size_gb": 3.6576,
  "head16mb_sha256": "d0b60b8e2132e2f3eae0551e51c34885fd6b92add874236eb534c4dd99207f2c",
  "head16mb_prefix": "d0b60b8e2132e2f3",
  "arch": "mistral3",
  "vocab_size": 131072,
  "bos_token": 1,
  "eos_token": 2,
  "reference_audio": {
    "path": "/content/better_speech_16k.wav",
    "sample_rate": 16000,
    "duration_s": 105.5,
    "peak_int16": 32768
  },
  "upload_role": "quantized_q8_0_gguf_artifact",
  "transcription_path_note": "Use VoxtralRealtimeForConditionalGeneration on HF/merged weights or an audio-capable realtime runtime for WAV transcription; text-only Llama() does not encode audio."
}
Saved pre-upload check: /content/voxtral_q8_preupload_check.json


In [ ]:
import glob
import os
import subprocess

GGUF_Q8 = "/content/drive/MyDrive/topogpt/gguf/voxtral-mini-4b-realtime-2602-topo-triton-Q8_0.gguf"
WAV_PATH = "/content/better_speech_16k.wav"

search_dirs = [
    "/content/drive/MyDrive/topogpt/llama.cpp/build/bin",
    "/content/llama.cpp/build/bin",
    "/usr/local/bin",
]

bins = []
for d in search_dirs:
    if os.path.isdir(d):
        for name in ["llama-mtmd-cli", "llama-cli", "llama-server"]:
            p = os.path.join(d, name)
            if os.path.isfile(p) and os.access(p, os.X_OK):
                bins.append(p)

print("Found binaries:", bins)
mmproj_candidates = glob.glob("/content/**/*mmproj*.gguf", recursive=True) + glob.glob(
    "/content/drive/MyDrive/topogpt/**/*mmproj*.gguf", recursive=True
)
print("Found mmproj candidates:", mmproj_candidates)

mtmd_bin = next((b for b in bins if os.path.basename(b) == "llama-mtmd-cli"), None)
if mtmd_bin and mmproj_candidates:
    cmd = [
        mtmd_bin,
        "-m",
        GGUF_Q8,
        "--mmproj",
        mmproj_candidates[0],
        "--audio",
        WAV_PATH,
        "-p",
        "Transcribe the speech accurately.",
    ]
    print("Running:", " ".join(cmd))
    res = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
    print("Return code:", res.returncode)
    print(res.stdout)
    if res.stderr:
        print("STDERR:", res.stderr[-1500:])
else:
    print("Missing either llama-mtmd-cli or an mmproj GGUF file in this environment.")
    print("To transcribe with Q8_0 GGUF here, build/install a llama.cpp version with Voxtral Realtime audio support and provide/extract the matching audio mmproj.")

In [ ]:
!chmod a+x /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-mtmd-cli
!ls -ltha /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-mtmd-cli

-rwx------ 1 root root 105K Nov  2 11:03 /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-mtmd-cli


In [ ]:
import glob
import subprocess

MTMD_CLI = "/content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-mtmd-cli"
GGUF_Q8 = "/content/drive/MyDrive/topogpt/gguf/voxtral-mini-4b-realtime-2602-topo-triton-Q8_0.gguf"
WAV_PATH = "/content/better_speech_16k.wav"

res = subprocess.run([MTMD_CLI, "--help"], capture_output=True, text=True)
help_text = (res.stdout or "") + "\n" + (res.stderr or "")
for line in help_text.splitlines():
    if any(k in line.lower() for k in ["audio", "mmproj", "-m ", "-p ", "prompt"]):
        print(line)

all_ggufs = glob.glob("/content/drive/MyDrive/topogpt/**/*.gguf", recursive=True) + glob.glob(
    "/content/*.gguf"
)
print("\nGGUF files found:", sorted(set(all_ggufs)))


GGUF files found: ['/content/drive/MyDrive/topogpt/deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf', '/content/drive/MyDrive/topogpt/deepseek_moe_16b_topocbp_q4_k_m.gguf', '/content/drive/MyDrive/topogpt/deepseek_r1_70b_topo_governed_merged/deepseek-r1-70b-topo-cbp-f16.gguf', '/content/drive/MyDrive/topogpt/gguf/nemotron/nemotron-3.5-bolt-e4b-topo-governed-fineweb-F16.gguf', '/content/drive/MyDrive/topogpt/gguf/nemotron/nemotron-3.5-bolt-e4b-topo-governed-fineweb-Q4_K_M.gguf', '/content/drive/MyDrive/topogpt/gguf/voxtral-mini-4b-realtime-2602-topo-triton-Q8_0.gguf', '/content/drive/MyDrive/topogpt/gguf/voxtral-mini-4b-realtime-2602-topo-triton-bf16.gguf', '/content/drive/MyDrive/topogpt/gguf_scratch/deepseek-r1-70b-topo-cbp-f16.gguf', '/content/drive/MyDrive/topogpt/llama.cpp/models/ggml-vocab-aquila.gguf', '/content/drive/MyDrive/topogpt/llama.cpp/models/ggml-vocab-baichuan.gguf', '/content/drive/MyDrive/topogpt/llama.cpp/models/ggml-vocab-bert-bge.gguf', '/content/drive/MyDriv

In [ ]:
!export LD_LIBRARY_PATH="/content/drive/MyDrive/topogpt/llama.cpp/build/bin:/content/drive/MyDrive/topogpt/llama.cpp/build/src:/content/drive/MyDrive/topogpt/llama.cpp/build/common:/content/drive/MyDrive/topogpt/llama.cpp/build/ggml/src:${LD_LIBRARY_PATH}"

!python3 /content/drive/MyDrive/topogpt/llama.cpp/convert_hf_to_gguf.py \
  /content/voxtral_merged_hf \
  --mmproj \
  --outtype f16 \
  --outfile /content/drive/MyDrive/topogpt/gguf/mmproj-voxtral-mini-4b-realtime-2602-f16.gguf

!/content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-mtmd-cli \
  -m /content/drive/MyDrive/topogpt/gguf/voxtral-mini-4b-realtime-2602-topo-triton-Q8_0.gguf \
  --mmproj /content/drive/MyDrive/topogpt/gguf/mmproj-voxtral-mini-4b-realtime-2602-f16.gguf \
  --audio /content/better_speech_16k.wav \
  -p "Transcribe the speech in this audio accurately."



/content# export LD_LIBRARY_PATH="/content/drive/MyDrive/topogpt/llama.cpp/build/bin:/content/drive/MyDrive/topogpt/llama.cpp/build/src:/content/drive/MyDrive/topogpt/llama.cpp/build/common:/content/drive/MyDrive/topogpt/llama.cpp/build/ggml/src:${LD_LIBRARY_PATH}"

python3 /content/drive/MyDrive/topogpt/llama.cpp/convert_hf_to_gguf.py \
  /content/voxtral_merged_hf \
  --mmproj \
  --outtype f16 \
  --outfile /content/drive/MyDrive/topogpt/gguf/mmproj-voxtral-mini-4b-realtime-2602-f16.gguf

/content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-mtmd-cli \
  -m /content/drive/MyDrive/topogpt/gguf/voxtral-mini-4b-realtime-2602-topo-triton-Q8_0.gguf \
  --mmproj /content/drive/MyDrive/topogpt/gguf/mmproj-voxtral-mini-4b-realtime-2602-f16.gguf \
  --audio /content/better_speech_16k.wav \
  -p "Transcribe the speech in this audio accurately."
INFO:hf-to-gguf:Loading model: voxtral_merged_hf
INFO:hf-to-gguf:Model architecture: VoxtralRealtimeForConditionalGeneration
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:a.conv1d.1.bias,                      torch.bfloat16 --> F32, shape = {1, 1280}
INFO:hf-to-gguf:a.conv1d.1.weight,                    torch.bfloat16 --> F16, shape = {3, 128, 1280}
INFO:hf-to-gguf:a.conv1d.2.bias,                      torch.bfloat16 --> F32, shape = {1, 1280}
INFO:hf-to-gguf:a.conv1d.2.weight,                    torch.bfloat16 --> F16, shape = {3, 1280, 1280}
INFO:hf-to-gguf:a.blk.0.ln2.weight,                   torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.0.ffn_down.bias,                torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.0.ffn_down.weight,              torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.0.ffn_gate.weight,              torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.0.ffn_up.weight,                torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.0.attn_k.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.0.attn_out.bias,                torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.0.attn_out.weight,              torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.0.attn_q.bias,                  torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.0.attn_q.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.0.attn_v.bias,                  torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.0.attn_v.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.0.ln1.weight,                   torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.1.ln2.weight,                   torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.1.ffn_down.bias,                torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.1.ffn_down.weight,              torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.1.ffn_gate.weight,              torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.1.ffn_up.weight,                torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.1.attn_k.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.1.attn_out.bias,                torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.1.attn_out.weight,              torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.1.attn_q.bias,                  torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.1.attn_q.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.1.attn_v.bias,                  torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.1.attn_v.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.1.ln1.weight,                   torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.10.ln2.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.10.ffn_down.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.10.ffn_down.weight,             torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.10.ffn_gate.weight,             torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.10.ffn_up.weight,               torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.10.attn_k.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.10.attn_out.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.10.attn_out.weight,             torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.10.attn_q.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.10.attn_q.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.10.attn_v.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.10.attn_v.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.10.ln1.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.11.ln2.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.11.ffn_down.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.11.ffn_down.weight,             torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.11.ffn_gate.weight,             torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.11.ffn_up.weight,               torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.11.attn_k.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.11.attn_out.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.11.attn_out.weight,             torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.11.attn_q.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.11.attn_q.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.11.attn_v.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.11.attn_v.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.11.ln1.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.12.ln2.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.12.ffn_down.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.12.ffn_down.weight,             torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.12.ffn_gate.weight,             torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.12.ffn_up.weight,               torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.12.attn_k.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.12.attn_out.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.12.attn_out.weight,             torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.12.attn_q.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.12.attn_q.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.12.attn_v.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.12.attn_v.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.12.ln1.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.13.ln2.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.13.ffn_down.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.13.ffn_down.weight,             torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.13.ffn_gate.weight,             torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.13.ffn_up.weight,               torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.13.attn_k.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.13.attn_out.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.13.attn_out.weight,             torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.13.attn_q.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.13.attn_q.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.13.attn_v.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.13.attn_v.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.13.ln1.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.14.ln2.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.14.ffn_down.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.14.ffn_down.weight,             torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.14.ffn_gate.weight,             torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.14.ffn_up.weight,               torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.14.attn_k.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.14.attn_out.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.14.attn_out.weight,             torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.14.attn_q.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.14.attn_q.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.14.attn_v.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.14.attn_v.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.14.ln1.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.15.ln2.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.15.ffn_down.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.15.ffn_down.weight,             torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.15.ffn_gate.weight,             torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.15.ffn_up.weight,               torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.15.attn_k.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.15.attn_out.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.15.attn_out.weight,             torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.15.attn_q.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.15.attn_q.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.15.attn_v.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.15.attn_v.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.15.ln1.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.16.ln2.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.16.ffn_down.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.16.ffn_down.weight,             torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.16.ffn_gate.weight,             torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.16.ffn_up.weight,               torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.16.attn_k.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.16.attn_out.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.16.attn_out.weight,             torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.16.attn_q.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.16.attn_q.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.16.attn_v.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.16.attn_v.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.16.ln1.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.17.ln2.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.17.ffn_down.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.17.ffn_down.weight,             torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.17.ffn_gate.weight,             torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.17.ffn_up.weight,               torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.17.attn_k.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.17.attn_out.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.17.attn_out.weight,             torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.17.attn_q.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.17.attn_q.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.17.attn_v.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.17.attn_v.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.17.ln1.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.18.ln2.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.18.ffn_down.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.18.ffn_down.weight,             torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.18.ffn_gate.weight,             torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.18.ffn_up.weight,               torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.18.attn_k.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.18.attn_out.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.18.attn_out.weight,             torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.18.attn_q.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.18.attn_q.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.18.attn_v.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.18.attn_v.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.18.ln1.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.19.ln2.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.19.ffn_down.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.19.ffn_down.weight,             torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.19.ffn_gate.weight,             torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.19.ffn_up.weight,               torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.19.attn_k.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.19.attn_out.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.19.attn_out.weight,             torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.19.attn_q.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.19.attn_q.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.19.attn_v.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.19.attn_v.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.19.ln1.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.2.ln2.weight,                   torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.2.ffn_down.bias,                torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.2.ffn_down.weight,              torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.2.ffn_gate.weight,              torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.2.ffn_up.weight,                torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.2.attn_k.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.2.attn_out.bias,                torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.2.attn_out.weight,              torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.2.attn_q.bias,                  torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.2.attn_q.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.2.attn_v.bias,                  torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.2.attn_v.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.2.ln1.weight,                   torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.20.ln2.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.20.ffn_down.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.20.ffn_down.weight,             torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.20.ffn_gate.weight,             torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.20.ffn_up.weight,               torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.20.attn_k.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.20.attn_out.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.20.attn_out.weight,             torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.20.attn_q.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.20.attn_q.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.20.attn_v.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.20.attn_v.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.20.ln1.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.21.ln2.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.21.ffn_down.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.21.ffn_down.weight,             torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.21.ffn_gate.weight,             torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.21.ffn_up.weight,               torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.21.attn_k.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.21.attn_out.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.21.attn_out.weight,             torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.21.attn_q.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.21.attn_q.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.21.attn_v.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.21.attn_v.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.21.ln1.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.22.ln2.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.22.ffn_down.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.22.ffn_down.weight,             torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.22.ffn_gate.weight,             torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.22.ffn_up.weight,               torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.22.attn_k.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.22.attn_out.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.22.attn_out.weight,             torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.22.attn_q.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.22.attn_q.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.22.attn_v.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.22.attn_v.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.22.ln1.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.23.ln2.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.23.ffn_down.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.23.ffn_down.weight,             torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.23.ffn_gate.weight,             torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.23.ffn_up.weight,               torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.23.attn_k.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.23.attn_out.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.23.attn_out.weight,             torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.23.attn_q.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.23.attn_q.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.23.attn_v.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.23.attn_v.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.23.ln1.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.24.ln2.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.24.ffn_down.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.24.ffn_down.weight,             torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.24.ffn_gate.weight,             torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.24.ffn_up.weight,               torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.24.attn_k.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.24.attn_out.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.24.attn_out.weight,             torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.24.attn_q.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.24.attn_q.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.24.attn_v.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.24.attn_v.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.24.ln1.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.25.ln2.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.25.ffn_down.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.25.ffn_down.weight,             torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.25.ffn_gate.weight,             torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.25.ffn_up.weight,               torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.25.attn_k.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.25.attn_out.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.25.attn_out.weight,             torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.25.attn_q.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.25.attn_q.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.25.attn_v.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.25.attn_v.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.25.ln1.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.26.ln2.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.26.ffn_down.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.26.ffn_down.weight,             torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.26.ffn_gate.weight,             torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.26.ffn_up.weight,               torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.26.attn_k.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.26.attn_out.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.26.attn_out.weight,             torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.26.attn_q.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.26.attn_q.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.26.attn_v.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.26.attn_v.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.26.ln1.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.27.ln2.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.27.ffn_down.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.27.ffn_down.weight,             torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.27.ffn_gate.weight,             torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.27.ffn_up.weight,               torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.27.attn_k.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.27.attn_out.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.27.attn_out.weight,             torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.27.attn_q.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.27.attn_q.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.27.attn_v.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.27.attn_v.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.27.ln1.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.28.ln2.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.28.ffn_down.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.28.ffn_down.weight,             torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.28.ffn_gate.weight,             torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.28.ffn_up.weight,               torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.28.attn_k.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.28.attn_out.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.28.attn_out.weight,             torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.28.attn_q.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.28.attn_q.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.28.attn_v.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.28.attn_v.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.28.ln1.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.29.ln2.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.29.ffn_down.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.29.ffn_down.weight,             torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.29.ffn_gate.weight,             torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.29.ffn_up.weight,               torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.29.attn_k.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.29.attn_out.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.29.attn_out.weight,             torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.29.attn_q.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.29.attn_q.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.29.attn_v.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.29.attn_v.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.29.ln1.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.3.ln2.weight,                   torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.3.ffn_down.bias,                torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.3.ffn_down.weight,              torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.3.ffn_gate.weight,              torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.3.ffn_up.weight,                torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.3.attn_k.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.3.attn_out.bias,                torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.3.attn_out.weight,              torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.3.attn_q.bias,                  torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.3.attn_q.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.3.attn_v.bias,                  torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.3.attn_v.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.3.ln1.weight,                   torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.30.ln2.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.30.ffn_down.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.30.ffn_down.weight,             torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.30.ffn_gate.weight,             torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.30.ffn_up.weight,               torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.30.attn_k.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.30.attn_out.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.30.attn_out.weight,             torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.30.attn_q.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.30.attn_q.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.30.attn_v.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.30.attn_v.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.30.ln1.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.31.ln2.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.31.ffn_down.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.31.ffn_down.weight,             torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.31.ffn_gate.weight,             torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.31.ffn_up.weight,               torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.31.attn_k.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.31.attn_out.bias,               torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.31.attn_out.weight,             torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.31.attn_q.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.31.attn_q.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.31.attn_v.bias,                 torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.31.attn_v.weight,               torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.31.ln1.weight,                  torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.4.ln2.weight,                   torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.4.ffn_down.bias,                torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.4.ffn_down.weight,              torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.4.ffn_gate.weight,              torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.4.ffn_up.weight,                torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.4.attn_k.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.4.attn_out.bias,                torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.4.attn_out.weight,              torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.4.attn_q.bias,                  torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.4.attn_q.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.4.attn_v.bias,                  torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.4.attn_v.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.4.ln1.weight,                   torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.5.ln2.weight,                   torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.5.ffn_down.bias,                torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.5.ffn_down.weight,              torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.5.ffn_gate.weight,              torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.5.ffn_up.weight,                torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.5.attn_k.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.5.attn_out.bias,                torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.5.attn_out.weight,              torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.5.attn_q.bias,                  torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.5.attn_q.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.5.attn_v.bias,                  torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.5.attn_v.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.5.ln1.weight,                   torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.6.ln2.weight,                   torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.6.ffn_down.bias,                torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.6.ffn_down.weight,              torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.6.ffn_gate.weight,              torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.6.ffn_up.weight,                torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.6.attn_k.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.6.attn_out.bias,                torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.6.attn_out.weight,              torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.6.attn_q.bias,                  torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.6.attn_q.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.6.attn_v.bias,                  torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.6.attn_v.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.6.ln1.weight,                   torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.7.ln2.weight,                   torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.7.ffn_down.bias,                torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.7.ffn_down.weight,              torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.7.ffn_gate.weight,              torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.7.ffn_up.weight,                torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.7.attn_k.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.7.attn_out.bias,                torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.7.attn_out.weight,              torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.7.attn_q.bias,                  torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.7.attn_q.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.7.attn_v.bias,                  torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.7.attn_v.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.7.ln1.weight,                   torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.8.ln2.weight,                   torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.8.ffn_down.bias,                torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.8.ffn_down.weight,              torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.8.ffn_gate.weight,              torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.8.ffn_up.weight,                torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.8.attn_k.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.8.attn_out.bias,                torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.8.attn_out.weight,              torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.8.attn_q.bias,                  torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.8.attn_q.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.8.attn_v.bias,                  torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.8.attn_v.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.8.ln1.weight,                   torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.9.ln2.weight,                   torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.9.ffn_down.bias,                torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.9.ffn_down.weight,              torch.bfloat16 --> F16, shape = {5120, 1280}
INFO:hf-to-gguf:a.blk.9.ffn_gate.weight,              torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.9.ffn_up.weight,                torch.bfloat16 --> F16, shape = {1280, 5120}
INFO:hf-to-gguf:a.blk.9.attn_k.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.9.attn_out.bias,                torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.blk.9.attn_out.weight,              torch.bfloat16 --> F16, shape = {2048, 1280}
INFO:hf-to-gguf:a.blk.9.attn_q.bias,                  torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.9.attn_q.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.9.attn_v.bias,                  torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:a.blk.9.attn_v.weight,                torch.bfloat16 --> F16, shape = {1280, 2048}
INFO:hf-to-gguf:a.blk.9.ln1.weight,                   torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:a.post_ln.weight,                     torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:mm.a.mlp.1.weight,                    torch.bfloat16 --> F16, shape = {5120, 3072}
INFO:hf-to-gguf:mm.a.mlp.2.weight,                    torch.bfloat16 --> F16, shape = {3072, 3072}
INFO:hf-to-gguf:Set meta model
INFO:hf-to-gguf:Set model parameters
INFO:hf-to-gguf:Set model quantization version
INFO:gguf.gguf_writer:Writing the following files:
INFO:gguf.gguf_writer:/content/drive/MyDrive/topogpt/gguf/mmproj-voxtral-mini-4b-realtime-2602-f16.gguf: n_tensors = 423, total_size = 2.0G
-realtime-2602-f16.gguf: n_tensors = 423, total_size = 2.0G
Writing: 100%|██████████████████████████████████████████████████████████████████| 1.99G/1.99G [00:11<00:00, 169Mbyte/s]
INFO:hf-to-gguf:Model successfully exported to /content/drive/MyDrive/topogpt/gguf/mmproj-voxtral-mini-4b-realtime-2602-f16.gguf
0.09.988.584 I cmn          init: llama threadpool init, n_threads = 6
0.17.206.046 W init_audio: audio input is in experimental stage and may have reduced quality:
    https://github.com/ggml-org/llama.cpp/discussions/13759
0.17.219.201 I mtmd_cli_context: chat template example:
[SYSTEM_PROMPT]You are a helpful assistant[/SYSTEM_PROMPT][INST]Hello[/INST]Hi there</s>[INST]How are you?[/INST]
0.17.219.210 I main: loading model: /content/drive/MyDrive/topogpt/gguf/voxtral-mini-4b-realtime-2602-topo-triton-Q8_0.gguf
Transcribing 105.5 s of audio, fed in chunks of 320 ms

Four score and seven years ago our fathers brought forth upon this continent a new nation conceived in liberty and dedicated to the proposition that all men are created equal now we are engaged in a great civil war testing whether that nation or any nation so conceived and so dedicated can long endure we are met on a great battlefield of that war we have come to dedicate a portion of it as a final resting place for those who died here that the nation might live this we may in all propriety do but in a larger sense we cannot dedicate we cannot consecrate we cannot hallow this ground the brave men living and dead who struggled here have hallowed it far above our poor power to add or detract the world will little note nor long remember what we say here while it can never forget what they did here it is rather for us the living we here be dedicated to the great task remaining before us that from these honored dead we take increased devotion to that cause for which they here gave the last full measure of devotion that we here highly resolve these dead shall not have died in vain that the nation shall have a new birth of freedom and that government of the people by the people for the people shall not perish from the earth

run_audio_stream: 1330 tokens, 105.50 s audio in 197.36 s (0.53x realtime)
run_audio_stream: encoder 70.50 s, decoder 126.85 s, 148.4 ms per 80 ms step
/content#
[0] 0:bash*                                       "82a0bf36c70e" 08:43 24-Nov-26

In [ ]:
!ls -ltha /content/drive/MyDrive/topogpt/gguf/

total 12G
drwx------  3 root root 4.0K Nov 24 08:32 .
-rw-------  1 root root 1.9G Nov 24 08:32 mmproj-voxtral-mini-4b-realtime-2602-f16.gguf
-rw-------  1 root root 3.5G Nov 24 07:05 voxtral-mini-4b-realtime-2602-topo-triton-Q8_0.gguf
-rw-------  1 root root 6.5G Nov 24 07:04 voxtral-mini-4b-realtime-2602-topo-triton-bf16.gguf
drwx------ 31 root root 4.0K Nov 24 01:12 ..
drwx------  2 root root 4.0K Nov 21 04:59 nemotron


## hf

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import HfApi, login

HF_TOKEN = userdata.get("HF_TOKEN")
user_or_name = "frankmorales2020"

login(token=HF_TOKEN)
api = HfApi(token=HF_TOKEN)

GGUF_DIR = "/content/drive/MyDrive/topogpt/gguf"
REPO_ID = f"{user_or_name}/voxtral-mini-4b-realtime-2602-topo-triton-GGUF"

FILES = [
    "voxtral-mini-4b-realtime-2602-topo-triton-Q8_0.gguf",
    "voxtral-mini-4b-realtime-2602-topo-triton-bf16.gguf",
    "mmproj-voxtral-mini-4b-realtime-2602-f16.gguf",
]

api.create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    private=False,
    exist_ok=True,
)

for filename in FILES:
    local_path = os.path.join(GGUF_DIR, filename)
    if not os.path.isfile(local_path):
        raise FileNotFoundError(f"Missing file: {local_path}")

    size_gb = os.path.getsize(local_path) / (1024**3)
    print(f"Uploading {filename} ({size_gb:.2f} GB) -> {REPO_ID}...")

    commit_info = api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=filename,
        repo_id=REPO_ID,
        repo_type="model",
        commit_message=f"Upload {filename}",
    )
    print(f"Uploaded {filename}: {commit_info.commit_url}")

Uploading voxtral-mini-4b-realtime-2602-topo-triton-Q8_0.gguf (3.41 GB) -> frankmorales2020/voxtral-mini-4b-realtime-2602-topo-triton-GGUF...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...602-topo-triton-Q8_0.gguf:   0%|          |  582kB / 3.66GB            

Validating                    :           |   0%            

Uploaded voxtral-mini-4b-realtime-2602-topo-triton-Q8_0.gguf: https://huggingface.co/frankmorales2020/voxtral-mini-4b-realtime-2602-topo-triton-GGUF/commit/055d1f2fb73e7e82e83df7480c113a11b7462a03
Uploading voxtral-mini-4b-realtime-2602-topo-triton-bf16.gguf (6.40 GB) -> frankmorales2020/voxtral-mini-4b-realtime-2602-topo-triton-GGUF...


... (6 outputs left out)

New Data Upload               : |          |  0.00B /  0.00B            

  ...4b-realtime-2602-f16.gguf:   1%|1         | 23.3MB / 1.99GB            

Validating                    :           |   0%            

Uploaded mmproj-voxtral-mini-4b-realtime-2602-f16.gguf: https://huggingface.co/frankmorales2020/voxtral-mini-4b-realtime-2602-topo-triton-GGUF/commit/e4e09b286861961a74b0f29ece88eaf97fea4245


## inference

In [ ]:
import os
import re
import json
import math
import time
import shutil
import hashlib
import logging
import pathlib
import warnings

# ---------------------------------------------------------------------------
# 1. Warning Suppression & Environment Setup (Before HF / PyTorch Imports)
# ---------------------------------------------------------------------------
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module=r".*transformers.*")
warnings.filterwarnings("ignore", message=r".*rope_scaling.*")
warnings.filterwarnings("ignore", message=r".*AttentionMaskConverter.*")
warnings.filterwarnings("ignore", message=r".*masking_utils.*")
warnings.filterwarnings("ignore", message=r".*use_return_dict.*")
warnings.filterwarnings("ignore", message=r".*torch_dtype.*")
warnings.filterwarnings("ignore", message=r".*resume_download.*")

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
for _offline_var in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE"):
    os.environ.pop(_offline_var, None)

for _lg_name in (
    "huggingface_hub",
    "huggingface_hub.utils._http",
    "transformers",
    "transformers.configuration_utils",
    "transformers.modeling_attn_mask_utils",
):
    logging.getLogger(_lg_name).setLevel(logging.ERROR)

In [ ]:
import os
import subprocess
from google.colab import userdata
from huggingface_hub import hf_hub_download, login

HF_TOKEN = userdata.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)

REPO_ID = "frankmorales2020/voxtral-mini-4b-realtime-2602-topo-triton-GGUF"
MODEL_FILENAME = "voxtral-mini-4b-realtime-2602-topo-triton-Q8_0.gguf"
MMPROJ_FILENAME = "mmproj-voxtral-mini-4b-realtime-2602-f16.gguf"

MTMD_CLI = "/content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-mtmd-cli"
WAV_PATH = "/content/better_speech_16k.wav"
CACHE_DIR = "/content/hf_gguf_test"

model_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=MODEL_FILENAME,
    local_dir=CACHE_DIR,
    token=HF_TOKEN,
)
mmproj_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=MMPROJ_FILENAME,
    local_dir=CACHE_DIR,
    token=HF_TOKEN,
)

env = os.environ.copy()
build_lib_dirs = [
    "/content/drive/MyDrive/topogpt/llama.cpp/build/bin",
    "/content/drive/MyDrive/topogpt/llama.cpp/build/src",
    "/content/drive/MyDrive/topogpt/llama.cpp/build/common",
    "/content/drive/MyDrive/topogpt/llama.cpp/build/ggml/src",
]
existing_ld = env.get("LD_LIBRARY_PATH", "")
env["LD_LIBRARY_PATH"] = ":".join(build_lib_dirs + ([existing_ld] if existing_ld else []))

cmd = [
    MTMD_CLI,
    "-m",
    model_path,
    "--mmproj",
    mmproj_path,
    "--audio",
    WAV_PATH,
    "-p",
    "Transcribe the speech in this audio accurately.",
]

print("Downloaded model:", model_path)
print("Downloaded mmproj:", mmproj_path)
print("Running:", " ".join(cmd))

proc = subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=600)
print("Exit code:", proc.returncode)
print("--- STDOUT ---")
print(proc.stdout)
if proc.stderr:
    print("--- STDERR ---")
    print(proc.stderr[-2000:])

Downloaded model: /content/hf_gguf_test/voxtral-mini-4b-realtime-2602-topo-triton-Q8_0.gguf
Downloaded mmproj: /content/hf_gguf_test/mmproj-voxtral-mini-4b-realtime-2602-f16.gguf
Running: /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-mtmd-cli -m /content/hf_gguf_test/voxtral-mini-4b-realtime-2602-topo-triton-Q8_0.gguf --mmproj /content/hf_gguf_test/mmproj-voxtral-mini-4b-realtime-2602-f16.gguf --audio /content/better_speech_16k.wav -p Transcribe the speech in this audio accurately.
Exit code: 0
--- STDOUT ---
Transcribing 105.5 s of audio, fed in chunks of 320 ms

Four score and seven years ago our fathers brought forth upon this continent a new nation conceived in liberty and dedicated to the proposition that all men are created equal now we are engaged in a great civil war testing whether that nation or any nation so conceived and so dedicated can long endure. We are met on a great battlefield of that war we have come to dedicate a portion of it as a final resting place fo

In [ ]:
import os
import re
import subprocess
from google.colab import userdata
from huggingface_hub import hf_hub_download, login

HF_TOKEN = userdata.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)

REPO_ID = "frankmorales2020/voxtral-mini-4b-realtime-2602-topo-triton-GGUF"
MODEL_FILENAME = "voxtral-mini-4b-realtime-2602-topo-triton-bf16.gguf"
MMPROJ_FILENAME = "mmproj-voxtral-mini-4b-realtime-2602-f16.gguf"

MTMD_CLI = "/content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-mtmd-cli"
WAV_PATH = "/content/better_speech_16k.wav"
CACHE_DIR = "/content/hf_gguf_test"

model_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=MODEL_FILENAME,
    local_dir=CACHE_DIR,
    token=HF_TOKEN,
)
mmproj_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=MMPROJ_FILENAME,
    local_dir=CACHE_DIR,
    token=HF_TOKEN,
)

env = os.environ.copy()
build_lib_dirs = [
    "/content/drive/MyDrive/topogpt/llama.cpp/build/bin",
    "/content/drive/MyDrive/topogpt/llama.cpp/build/src",
    "/content/drive/MyDrive/topogpt/llama.cpp/build/common",
    "/content/drive/MyDrive/topogpt/llama.cpp/build/ggml/src",
]
existing_ld = env.get("LD_LIBRARY_PATH", "")
env["LD_LIBRARY_PATH"] = ":".join(build_lib_dirs + ([existing_ld] if existing_ld else []))

cmd = [
    MTMD_CLI,
    "-m",
    model_path,
    "--mmproj",
    mmproj_path,
    "--audio",
    WAV_PATH,
    "-p",
    "Transcribe the speech in this audio accurately.",
]

print("Downloaded model:", model_path)
print("Downloaded mmproj:", mmproj_path)
print("Running:", " ".join(cmd))

proc = subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=900)
print("Exit code:", proc.returncode)
print("--- STDOUT ---")
print(proc.stdout)
if proc.stderr:
    print("--- STDERR ---")
    print(proc.stderr[-2000:])

# Parse BF16 timing metrics and compare with Q8_0 baseline on /content/better_speech_16k.wav
q8_metrics = {
    "size_gb": 3.66,
    "total_s": 203.52,
    "rt_factor": 0.52,
    "encoder_s": 71.73,
    "decoder_s": 131.77,
    "ms_per_80ms": 153.0,
}

bf16_size_gb = os.path.getsize(model_path) / (1024**3)
m_total = re.search(
    r"run_audio_stream:\s+(\d+)\s+tokens,\s+([\d.]+)\s+s audio in\s+([\d.]+)\s+s\s+\(([\d.]+)x realtime\)",
    proc.stdout,
)
m_parts = re.search(
    r"run_audio_stream:\s+encoder\s+([\d.]+)\s+s,\s+decoder\s+([\d.]+)\s+s,\s+([\d.]+)\s+ms per 80 ms step",
    proc.stdout,
)

if m_total and m_parts:
    bf16_total_s = float(m_total.group(3))
    bf16_rt = float(m_total.group(4))
    bf16_enc_s = float(m_parts.group(1))
    bf16_dec_s = float(m_parts.group(2))
    bf16_step_ms = float(m_parts.group(3))

    print("\n=== Runtime Comparison (105.50 s audio) ===")
    print(f"File size:    Q8_0 = {q8_metrics['size_gb']:.2f} GB  |  BF16 = {bf16_size_gb:.2f} GB")
    print(f"Encoder (F16): Q8_0 = {q8_metrics['encoder_s']:.2f} s   |  BF16 = {bf16_enc_s:.2f} s")
    print(f"Decoder:      Q8_0 = {q8_metrics['decoder_s']:.2f} s  |  BF16 = {bf16_dec_s:.2f} s")
    print(f"Total time:   Q8_0 = {q8_metrics['total_s']:.2f} s  |  BF16 = {bf16_total_s:.2f} s")
    print(f"Speed ratio:  Q8_0 = {q8_metrics['rt_factor']:.2f}x RT   |  BF16 = {bf16_rt:.2f}x RT")
    print(f"Step latency: Q8_0 = {q8_metrics['ms_per_80ms']:.1f} ms/80ms |  BF16 = {bf16_step_ms:.1f} ms/80ms")

voxtral-mini-4b-realtime-2602-topo-trito(…): reconstructing file:   0%|          |  0.00B / 6.88GB            

voxtral-mini-4b-realtime-2602-topo-trito(…): downloading bytes:           |  0.00B            

Downloaded model: /content/hf_gguf_test/voxtral-mini-4b-realtime-2602-topo-triton-bf16.gguf
Downloaded mmproj: /content/hf_gguf_test/mmproj-voxtral-mini-4b-realtime-2602-f16.gguf
Running: /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-mtmd-cli -m /content/hf_gguf_test/voxtral-mini-4b-realtime-2602-topo-triton-bf16.gguf --mmproj /content/hf_gguf_test/mmproj-voxtral-mini-4b-realtime-2602-f16.gguf --audio /content/better_speech_16k.wav -p Transcribe the speech in this audio accurately.
Exit code: 0
--- STDOUT ---
Transcribing 105.5 s of audio, fed in chunks of 320 ms

Four score and seven years ago our fathers brought forth upon this continent a new nation conceived in liberty and dedicated to the proposition that all men are created equal now we are engaged in a great civil war testing whether that nation or any nation so conceived and so dedicated can long endure we are met on a great battlefield of that war we have come to dedicate a portion of it as a final resting place for

Both runs finished cleanly with exit code `0`, producing the exact same `1,330`-token Gettysburg Address transcript across `105.50 s` of audio in `320 ms` chunks.

### `Q8_0` vs. `BF16` Benchmark (`105.50 s` Audio, `1,330` Tokens)

| Metric | `Q8_0` | `BF16` | Difference (`BF16` vs. `Q8_0`) |
| --- | --- | --- | --- |
| **Decoder GGUF Size** | `3.66 GB` | `6.88 GB` (`6.40 GiB`) | `+3.22 GB` (`~1.88x` larger) |
| **Audio Encoder (`F16` `mmproj`)** | `71.73 s` | `71.39 s` | `-0.34 s` (virtually identical) |
| **Text Decoder Time** | `131.77 s` | `192.49 s` | `+60.72 s` (`+46.1%` slower) |
| **Total Wall Time** | `203.52 s` | `263.90 s` | `+60.38 s` (`+29.7%` slower) |
| **Realtime Factor** | `0.52x` | `0.40x` | `-0.12x` |
| **Step Latency (`per 80 ms`)** | `153.0 ms` | `198.4 ms` | `+45.4 ms` (`+29.7%`) |

### Takeaways

* **Encoder is unchanged:** `mmproj-voxtral-mini-4b-realtime-2602-f16.gguf` takes right around `71.4–71.7 s` in both runs.
* **`Q8_0` is the sweet spot here:** It cuts decoder runtime by about a minute (`131.77 s` vs. `192.49 s`) and keeps the decoder footprint at `3.66 GB` while returning the same transcription on this file.